In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F
from pyspark.sql import Window

In [0]:
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
FECHA_INICIO = "2016-01-01 00:00:00"
FECHA_FIN = "2016-07-01 00:00:00"

print(f"Fecha inicio : {FECHA_INICIO}")
print(f"Fecha fin    : {FECHA_FIN}")

Fecha inicio : 2016-01-01 00:00:00
Fecha fin    : 2016-07-01 00:00:00


In [0]:
def borough_categoria(borough):
    if borough is None:
        return "SIN_BOROUGH"

    borough_normalizado = borough.strip().upper()

    if borough_normalizado == "MANHATTAN":
        return "MANHATTAN"

    elif borough_normalizado in (
        "BRONX",
        "BROOKLYN",
        "QUEENS",
        "STATEN ISLAND"
    ):
        return "OUTER_BOROUGH"

    elif borough_normalizado == "EWR":
        return "OUTSIDE_NYC"

    else:
        return "OTHER"

In [0]:
borough_categoria_udf = F.udf(
    borough_categoria,
    StringType()
)

In [0]:
df_taxi = spark.table(
    f"{catalogo}.{esquema_source}.taxi_trips"
)

df_citibike = spark.table(
    f"{catalogo}.{esquema_source}.citibike_trips"
)

df_weather = spark.table(
    f"{catalogo}.{esquema_source}.weather_hourly"
)

df_taxi_zones_geojson = spark.table(
    f"{catalogo}.{esquema_source}.taxi_zones_geojson"
)

df_taxi_zones_csv = spark.table(
    f"{catalogo}.{esquema_source}.taxi_zones_csv"
)

print("Tablas Bronze cargadas correctamente.")

Tablas Bronze cargadas correctamente.


In [0]:
print("=== REGISTROS BRONZE ===")

print(
    f"taxi_trips           : {df_taxi.count():,}"
)

print(
    f"citibike_trips       : {df_citibike.count():,}"
)

print(
    f"weather_hourly       : {df_weather.count():,}"
)

print(
    f"taxi_zones_geojson   : {df_taxi_zones_geojson.count():,}"
)

print(
    f"taxi_zones_csv       : {df_taxi_zones_csv.count():,}"
)

=== REGISTROS BRONZE ===
taxi_trips           : 1,458,644
citibike_trips       : 5,676,020
weather_hourly       : 59,760
taxi_zones_geojson   : 263
taxi_zones_csv       : 263


In [0]:
df_taxi_clean = (
    df_taxi
    .dropna(how="all")
    .filter(
        col("id").isNotNull()
        & col("pickup_datetime").isNotNull()
        & col("dropoff_datetime").isNotNull()
    )
    .filter(
        (col("pickup_datetime") >= lit(FECHA_INICIO).cast("timestamp"))
        & (col("pickup_datetime") < lit(FECHA_FIN).cast("timestamp"))
    )
)

total_taxi_bronze = df_taxi.count()
total_taxi_clean = df_taxi_clean.count()

print(f"Taxi Bronze        : {total_taxi_bronze:,}")
print(f"Taxi después filtro: {total_taxi_clean:,}")
print(
    f"Registros excluidos: "
    f"{total_taxi_bronze - total_taxi_clean:,}"
)

Taxi Bronze        : 1,458,644
Taxi después filtro: 1,458,644
Registros excluidos: 0


In [0]:
df_taxi_dq = (
    df_taxi_clean

    # ============================================================
    # VALIDACIÓN DE DURACIÓN
    # ============================================================

    .withColumn(
        "dq_invalid_duration",
        when(
            col("trip_duration").isNull()
            | (col("trip_duration") <= 0)
            | (col("dropoff_datetime") <= col("pickup_datetime")),
            lit(1)
        ).otherwise(lit(0))
    )

    # ============================================================
    # PASAJEROS
    # ============================================================

    .withColumn(
        "dq_invalid_passenger_count",
        when(
            col("passenger_count").isNull()
            | (col("passenger_count") <= 0),
            lit(1)
        ).otherwise(lit(0))
    )

    # ============================================================
    # COORDENADAS PICKUP
    # ============================================================

    .withColumn(
        "dq_invalid_pickup_coordinates",
        when(
            col("pickup_latitude").isNull()
            | col("pickup_longitude").isNull()
            | (col("pickup_latitude") < -90)
            | (col("pickup_latitude") > 90)
            | (col("pickup_longitude") < -180)
            | (col("pickup_longitude") > 180)
            | (
                (col("pickup_latitude") == 0)
                & (col("pickup_longitude") == 0)
            ),
            lit(1)
        ).otherwise(lit(0))
    )

    # ============================================================
    # COORDENADAS DROPOFF
    # ============================================================

    .withColumn(
        "dq_invalid_dropoff_coordinates",
        when(
            col("dropoff_latitude").isNull()
            | col("dropoff_longitude").isNull()
            | (col("dropoff_latitude") < -90)
            | (col("dropoff_latitude") > 90)
            | (col("dropoff_longitude") < -180)
            | (col("dropoff_longitude") > 180)
            | (
                (col("dropoff_latitude") == 0)
                & (col("dropoff_longitude") == 0)
            ),
            lit(1)
        ).otherwise(lit(0))
    )

    # ============================================================
    # DURACIÓN EXTREMA
    # ============================================================

    .withColumn(
        "dq_long_duration",
        when(
            col("trip_duration") > 4 * 60 * 60,
            lit(1)
        ).otherwise(lit(0))
    )
)

In [0]:
df_taxi_dq = df_taxi_dq.withColumn(
    "dq_valid_trip",
    when(
        (col("dq_invalid_duration") == 0)
        & (col("dq_invalid_passenger_count") == 0)
        & (col("dq_invalid_pickup_coordinates") == 0)
        & (col("dq_invalid_dropoff_coordinates") == 0),
        lit(1)
    ).otherwise(lit(0))
)

In [0]:
df_taxi_dq.select(
    count("*").alias("total_registros"),

    sum("dq_invalid_duration")
        .alias("invalid_duration"),

    sum("dq_invalid_passenger_count")
        .alias("invalid_passenger_count"),

    sum("dq_invalid_pickup_coordinates")
        .alias("invalid_pickup_coordinates"),

    sum("dq_invalid_dropoff_coordinates")
        .alias("invalid_dropoff_coordinates"),

    sum("dq_long_duration")
        .alias("long_duration"),

    sum(
        when(
            col("dq_valid_trip") == 1,
            lit(1)
        ).otherwise(lit(0))
    ).alias("valid_trips")

).show()

+---------------+----------------+-----------------------+--------------------------+---------------------------+-------------+-----------+
|total_registros|invalid_duration|invalid_passenger_count|invalid_pickup_coordinates|invalid_dropoff_coordinates|long_duration|valid_trips|
+---------------+----------------+-----------------------+--------------------------+---------------------------+-------------+-----------+
|        1458644|               0|                     60|                         0|                          0|         2077|    1458584|
+---------------+----------------+-----------------------+--------------------------+---------------------------+-------------+-----------+



In [0]:
df_taxi_transform = (
    df_taxi_dq

    .withColumn(
        "duration_minutes",
        col("trip_duration") / lit(60.0)
    )

    .withColumn(
        "duration_hours",
        col("trip_duration") / lit(3600.0)
    )

    .withColumn(
        "pickup_date",
        to_date(col("pickup_datetime"))
    )

    .withColumn(
        "pickup_hour",
        hour(col("pickup_datetime"))
    )

    .withColumn(
        "pickup_day_of_week",
        date_format(
            col("pickup_datetime"),
            "EEEE"
        )
    )

    .withColumn(
        "pickup_month",
        month(col("pickup_datetime"))
    )

    .withColumn(
        "pickup_year",
        year(col("pickup_datetime"))
    )

    .withColumn(
        "is_weekend",
        when(
            dayofweek(col("pickup_datetime")).isin(1, 7),
            lit(1)
        ).otherwise(lit(0))
    )
)

In [0]:
df_taxi_transform.select(
    min("pickup_datetime").alias("pickup_min"),
    max("pickup_datetime").alias("pickup_max"),

    min("duration_minutes").alias("duration_min_minutes"),
    max("duration_minutes").alias("duration_max_minutes"),

    min("pickup_hour").alias("hour_min"),
    max("pickup_hour").alias("hour_max"),

    countDistinct("pickup_month").alias("meses"),
    countDistinct("pickup_year").alias("anios")
).show()

+-------------------+-------------------+--------------------+--------------------+--------+--------+-----+-----+
|         pickup_min|         pickup_max|duration_min_minutes|duration_max_minutes|hour_min|hour_max|meses|anios|
+-------------------+-------------------+--------------------+--------------------+--------+--------+-----+-----+
|2016-01-01 00:00:17|2016-06-30 23:59:39|0.016666666666666666|   58771.36666666667|       0|      23|    6|    1|
+-------------------+-------------------+--------------------+--------------------+--------+--------+-----+-----+



In [0]:
df_taxi_silver = df_taxi_transform.select(
    col("id"),
    col("vendor_id"),
    col("pickup_datetime"),
    col("dropoff_datetime"),
    col("passenger_count"),
    col("pickup_longitude"),
    col("pickup_latitude"),
    col("dropoff_longitude"),
    col("dropoff_latitude"),
    col("store_and_fwd_flag"),
    col("trip_duration"),

    col("_ingestion_timestamp"),
    col("_source_file"),
    col("_source_system"),
    col("_batch_id"),

    col("dq_invalid_duration"),
    col("dq_invalid_passenger_count"),
    col("dq_invalid_pickup_coordinates"),
    col("dq_invalid_dropoff_coordinates"),
    col("dq_long_duration"),
    col("dq_valid_trip"),

    col("duration_minutes"),
    col("duration_hours"),
    col("pickup_date"),
    col("pickup_hour"),
    col("pickup_day_of_week"),
    col("pickup_month"),
    col("pickup_year"),
    col("is_weekend")
)

In [0]:
tabla_taxi_silver = (
    f"{catalogo}.{esquema_sink}.taxi_trips"
)

print("=== SCHEMA DATAFRAME TAXI SILVER ===")
df_taxi_silver.printSchema()

print("=== SCHEMA TABLA TAXI SILVER ===")
spark.table(tabla_taxi_silver).printSchema()

columnas_df = df_taxi_silver.columns
columnas_tabla = spark.table(tabla_taxi_silver).columns

print(f"Columnas DataFrame : {len(columnas_df)}")
print(f"Columnas tabla     : {len(columnas_tabla)}")

if columnas_df != columnas_tabla:
    raise Exception(
        "El orden o nombre de las columnas del DataFrame "
        "no coincide con la tabla Silver."
    )

print("Contrato de columnas correcto.")

=== SCHEMA DATAFRAME TAXI SILVER ===
root
 |-- id: string (nullable = true)
 |-- vendor_id: integer (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- trip_duration: long (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_system: string (nullable = true)
 |-- _batch_id: string (nullable = true)
 |-- dq_invalid_duration: integer (nullable = false)
 |-- dq_invalid_passenger_count: integer (nullable = false)
 |-- dq_invalid_pickup_coordinates: integer (nullable = false)
 |-- dq_invalid_dropoff_coordinates: integer (nullable = false)
 |-- dq_long_duration:

In [0]:
df_taxi_silver.write.mode("overwrite")\
    .insertInto(tabla_taxi_silver)

In [0]:
total_taxi_bronze = df_taxi.count()
total_taxi_transform = df_taxi_silver.count()
total_taxi_silver = spark.table(tabla_taxi_silver).count()

print(f"Taxi Bronze     : {total_taxi_bronze:,}")
print(f"Taxi Transform  : {total_taxi_transform:,}")
print(f"Taxi Silver     : {total_taxi_silver:,}")

if total_taxi_transform != total_taxi_silver:
    raise Exception(
        "La cantidad de registros Transform y Silver no coincide."
    )

print("Transformación Taxi finalizada correctamente.")

Taxi Bronze     : 1,458,644
Taxi Transform  : 1,458,644
Taxi Silver     : 1,458,644
Transformación Taxi finalizada correctamente.


In [0]:
df_taxi_silver_check = spark.table(
    tabla_taxi_silver
)

df_taxi_silver_check.select(
    count("*").alias("total_registros"),

    sum("dq_invalid_duration")
        .alias("invalid_duration"),

    sum("dq_invalid_passenger_count")
        .alias("invalid_passenger_count"),

    sum("dq_invalid_pickup_coordinates")
        .alias("invalid_pickup_coordinates"),

    sum("dq_invalid_dropoff_coordinates")
        .alias("invalid_dropoff_coordinates"),

    sum("dq_long_duration")
        .alias("long_duration"),

    sum(
        when(
            col("dq_valid_trip") == 1,
            lit(1)
        ).otherwise(lit(0))
    ).alias("valid_trips")

).show()

+---------------+----------------+-----------------------+--------------------------+---------------------------+-------------+-----------+
|total_registros|invalid_duration|invalid_passenger_count|invalid_pickup_coordinates|invalid_dropoff_coordinates|long_duration|valid_trips|
+---------------+----------------+-----------------------+--------------------------+---------------------------+-------------+-----------+
|        1458644|               0|                     60|                         0|                          0|         2077|    1458584|
+---------------+----------------+-----------------------+--------------------------+---------------------------+-------------+-----------+



In [0]:
df_citibike_clean = (
    df_citibike
    .dropna(how="all")
    .filter(
        col("starttime").isNotNull()
        & col("stoptime").isNotNull()
    )
    .filter(
        (col("starttime") >= lit(FECHA_INICIO).cast("timestamp"))
        & (col("starttime") < lit(FECHA_FIN).cast("timestamp"))
    )
)

total_citibike_bronze = df_citibike.count()
total_citibike_clean = df_citibike_clean.count()

print(f"Citi Bike Bronze        : {total_citibike_bronze:,}")
print(f"Citi Bike después filtro: {total_citibike_clean:,}")
print(
    f"Registros excluidos     : "
    f"{total_citibike_bronze - total_citibike_clean:,}"
)

Citi Bike Bronze        : 5,676,020
Citi Bike después filtro: 5,676,020
Registros excluidos     : 0


In [0]:
df_citibike_dq = (
    df_citibike_clean

    # ============================================================
    # DURACIÓN
    # ============================================================

    .withColumn(
        "dq_invalid_duration",
        when(
            col("tripduration").isNull()
            | (col("tripduration") <= 0)
            | (col("stoptime") <= col("starttime")),
            lit(1)
        ).otherwise(lit(0))
    )

    # Diferencia entre duración calculada con timestamps
    # y duración proporcionada por Citi Bike.
    .withColumn(
        "dq_duration_mismatch",
        when(
            abs(
                (
                    col("stoptime").cast("long")
                    - col("starttime").cast("long")
                )
                - col("tripduration")
            ) > 1,
            lit(1)
        ).otherwise(lit(0))
    )

    # ============================================================
    # COORDENADAS ESTACIÓN INICIO
    # ============================================================

    .withColumn(
        "dq_invalid_start_coordinates",
        when(
            col("start_station_latitude").isNull()
            | col("start_station_longitude").isNull()
            | (col("start_station_latitude") < -90)
            | (col("start_station_latitude") > 90)
            | (col("start_station_longitude") < -180)
            | (col("start_station_longitude") > 180)
            | (
                (col("start_station_latitude") == 0)
                & (col("start_station_longitude") == 0)
            ),
            lit(1)
        ).otherwise(lit(0))
    )

    # ============================================================
    # COORDENADAS ESTACIÓN FIN
    # ============================================================

    .withColumn(
        "dq_invalid_end_coordinates",
        when(
            col("end_station_latitude").isNull()
            | col("end_station_longitude").isNull()
            | (col("end_station_latitude") < -90)
            | (col("end_station_latitude") > 90)
            | (col("end_station_longitude") < -180)
            | (col("end_station_longitude") > 180)
            | (
                (col("end_station_latitude") == 0)
                & (col("end_station_longitude") == 0)
            ),
            lit(1)
        ).otherwise(lit(0))
    )

    # ============================================================
    # INFORMACIÓN DEMOGRÁFICA
    # ============================================================

    .withColumn(
        "dq_missing_birth_year",
        when(
            col("birth_year").isNull(),
            lit(1)
        ).otherwise(lit(0))
    )

    # ============================================================
    # DURACIÓN EXTREMA
    # ============================================================

    .withColumn(
        "dq_long_duration",
        when(
            col("tripduration") > 4 * 60 * 60,
            lit(1)
        ).otherwise(lit(0))
    )
)

In [0]:
df_citibike_transform = (
    df_citibike_dq

    # ============================================================
    # ID DETERMINÍSTICO DEL VIAJE
    # ============================================================

    .withColumn(
        "trip_id",
        sha2(
            concat_ws(
                "||",
                coalesce(col("_source_file"), lit("<NULL>")),
                coalesce(col("bikeid").cast("string"), lit("<NULL>")),
                coalesce(col("starttime").cast("string"), lit("<NULL>")),
                coalesce(col("stoptime").cast("string"), lit("<NULL>")),
                coalesce(col("start_station_id").cast("string"), lit("<NULL>")),
                coalesce(col("end_station_id").cast("string"), lit("<NULL>")),
                coalesce(col("tripduration").cast("string"), lit("<NULL>"))
            ),
            256
        )
    )

    # ============================================================
    # DURACIÓN
    # ============================================================

    .withColumn(
        "duration_minutes",
        col("tripduration") / lit(60.0)
    )

    .withColumn(
        "duration_hours",
        col("tripduration") / lit(3600.0)
    )

    # ============================================================
    # VARIABLES TEMPORALES
    # ============================================================

    .withColumn(
        "start_date",
        to_date(col("starttime"))
    )

    .withColumn(
        "start_hour",
        hour(col("starttime"))
    )

    .withColumn(
        "start_day_of_week",
        date_format(
            col("starttime"),
            "EEEE"
        )
    )

    .withColumn(
        "start_month",
        month(col("starttime"))
    )

    .withColumn(
        "start_year",
        year(col("starttime"))
    )

    .withColumn(
        "is_weekend",
        when(
            dayofweek(col("starttime")).isin(1, 7),
            lit(1)
        ).otherwise(lit(0))
    )

    # ============================================================
    # EDAD DEL USUARIO
    # ============================================================

    .withColumn(
        "rider_age",
        when(
            col("birth_year").isNotNull(),
            year(col("starttime")) - col("birth_year")
        )
    )

    .withColumn(
        "dq_age_outlier",
        when(
            col("rider_age").isNotNull()
            & (
                (col("rider_age") <= 0)
                | (col("rider_age") > 100)
            ),
            lit(1)
        ).otherwise(lit(0))
    )

    # ============================================================
    # ESTANDARIZACIÓN USER TYPE
    # ============================================================

    .withColumn(
        "usertype",
        upper(trim(col("usertype")))
    )
)

In [0]:
df_citibike_transform = df_citibike_transform.withColumn(
    "dq_valid_trip",
    when(
        (col("dq_invalid_duration") == 0)
        & (col("dq_invalid_start_coordinates") == 0)
        & (col("dq_invalid_end_coordinates") == 0),
        lit(1)
    ).otherwise(lit(0))
)

In [0]:
df_citibike_transform.select(
    count("*").alias("total_registros"),

    sum("dq_invalid_duration")
        .alias("invalid_duration"),

    sum("dq_duration_mismatch")
        .alias("duration_mismatch"),

    sum("dq_invalid_start_coordinates")
        .alias("invalid_start_coordinates"),

    sum("dq_invalid_end_coordinates")
        .alias("invalid_end_coordinates"),

    sum("dq_missing_birth_year")
        .alias("missing_birth_year"),

    sum("dq_age_outlier")
        .alias("age_outlier"),

    sum("dq_long_duration")
        .alias("long_duration"),

    sum(
        when(
            col("dq_valid_trip") == 1,
            lit(1)
        ).otherwise(lit(0))
    ).alias("valid_trips"),

    countDistinct("trip_id")
        .alias("trip_id_unicos")

).show()

+---------------+----------------+-----------------+-------------------------+-----------------------+------------------+-----------+-------------+-----------+--------------+
|total_registros|invalid_duration|duration_mismatch|invalid_start_coordinates|invalid_end_coordinates|missing_birth_year|age_outlier|long_duration|valid_trips|trip_id_unicos|
+---------------+----------------+-----------------+-------------------------+-----------------------+------------------+-----------+-------------+-----------+--------------+
|        5676020|               0|              127|                        0|                     74|            649611|       2075|        13466|    5675946|       5676020|
+---------------+----------------+-----------------+-------------------------+-----------------------+------------------+-----------+-------------+-----------+--------------+



In [0]:
df_citibike_transform.select(
    min("starttime").alias("start_min"),
    max("starttime").alias("start_max"),

    min("duration_minutes").alias("duration_min_minutes"),
    max("duration_minutes").alias("duration_max_minutes"),

    min("start_hour").alias("hour_min"),
    max("start_hour").alias("hour_max"),

    countDistinct("start_month").alias("meses"),
    countDistinct("start_year").alias("anios"),

    count("*").alias("total_registros"),
    countDistinct("trip_id").alias("trip_id_unicos")
).show()

+-------------------+-------------------+--------------------+--------------------+--------+--------+-----+-----+---------------+--------------+
|          start_min|          start_max|duration_min_minutes|duration_max_minutes|hour_min|hour_max|meses|anios|total_registros|trip_id_unicos|
+-------------------+-------------------+--------------------+--------------------+--------+--------+-----+-----+---------------+--------------+
|2016-01-01 00:00:41|2016-06-30 23:59:58|  1.0166666666666666|   60879.11666666667|       0|      23|    6|    1|        5676020|       5676020|
+-------------------+-------------------+--------------------+--------------------+--------+--------+-----+-----+---------------+--------------+



In [0]:
df_citibike_silver = df_citibike_transform.select(
    col("trip_id"),

    col("tripduration"),
    col("starttime"),
    col("stoptime"),

    col("start_station_id"),
    col("start_station_name"),
    col("start_station_latitude"),
    col("start_station_longitude"),

    col("end_station_id"),
    col("end_station_name"),
    col("end_station_latitude"),
    col("end_station_longitude"),

    col("bikeid"),
    col("usertype"),
    col("birth_year"),
    col("gender"),

    col("_ingestion_timestamp"),
    col("_source_file"),
    col("_source_system"),
    col("_batch_id"),

    col("dq_invalid_duration"),
    col("dq_duration_mismatch"),
    col("dq_invalid_start_coordinates"),
    col("dq_invalid_end_coordinates"),
    col("dq_missing_birth_year"),
    col("dq_age_outlier"),
    col("dq_long_duration"),
    col("dq_valid_trip"),

    col("duration_minutes"),
    col("duration_hours"),

    col("start_date"),
    col("start_hour"),
    col("start_day_of_week"),
    col("start_month"),
    col("start_year"),
    col("is_weekend"),

    col("rider_age")
)

In [0]:
tabla_citibike_silver = (
    f"{catalogo}.{esquema_sink}.citibike_trips"
)

print("=== SCHEMA DATAFRAME CITIBIKE SILVER ===")
df_citibike_silver.printSchema()

print("=== SCHEMA TABLA CITIBIKE SILVER ===")
spark.table(tabla_citibike_silver).printSchema()

columnas_df = df_citibike_silver.columns
columnas_tabla = spark.table(tabla_citibike_silver).columns

print(f"Columnas DataFrame : {len(columnas_df)}")
print(f"Columnas tabla     : {len(columnas_tabla)}")

if columnas_df != columnas_tabla:
    raise Exception(
        "El orden o nombre de las columnas del DataFrame "
        "no coincide con la tabla Silver."
    )

print("Contrato de columnas Citi Bike correcto.")

=== SCHEMA DATAFRAME CITIBIKE SILVER ===
root
 |-- trip_id: string (nullable = true)
 |-- tripduration: long (nullable = true)
 |-- starttime: timestamp (nullable = true)
 |-- stoptime: timestamp (nullable = true)
 |-- start_station_id: integer (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_latitude: double (nullable = true)
 |-- start_station_longitude: double (nullable = true)
 |-- end_station_id: integer (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_latitude: double (nullable = true)
 |-- end_station_longitude: double (nullable = true)
 |-- bikeid: integer (nullable = true)
 |-- usertype: string (nullable = true)
 |-- birth_year: integer (nullable = true)
 |-- gender: integer (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_system: string (nullable = true)
 |-- _batch_id: string (nullable = true)
 |-- dq_invalid_duration: integ

In [0]:
df_citibike_silver.write.mode("overwrite")\
    .insertInto(tabla_citibike_silver)

In [0]:
total_citibike_bronze = df_citibike.count()
total_citibike_transform = df_citibike_silver.count()
total_citibike_silver = spark.table(
    tabla_citibike_silver
).count()

print(f"Citi Bike Bronze    : {total_citibike_bronze:,}")
print(f"Citi Bike Transform : {total_citibike_transform:,}")
print(f"Citi Bike Silver    : {total_citibike_silver:,}")

if total_citibike_transform != total_citibike_silver:
    raise Exception(
        "La cantidad de registros Transform y Silver no coincide."
    )

print("Transformación Citi Bike finalizada correctamente.")

Citi Bike Bronze    : 5,676,020
Citi Bike Transform : 5,676,020
Citi Bike Silver    : 5,676,020
Transformación Citi Bike finalizada correctamente.


In [0]:
df_citibike_silver_check = spark.table(
    tabla_citibike_silver
)

df_citibike_silver_check.select(
    count("*").alias("total_registros"),

    sum("dq_invalid_duration")
        .alias("invalid_duration"),

    sum("dq_duration_mismatch")
        .alias("duration_mismatch"),

    sum("dq_invalid_start_coordinates")
        .alias("invalid_start_coordinates"),

    sum("dq_invalid_end_coordinates")
        .alias("invalid_end_coordinates"),

    sum("dq_missing_birth_year")
        .alias("missing_birth_year"),

    sum("dq_age_outlier")
        .alias("age_outlier"),

    sum("dq_long_duration")
        .alias("long_duration"),

    sum(
        when(
            col("dq_valid_trip") == 1,
            lit(1)
        ).otherwise(lit(0))
    ).alias("valid_trips"),

    countDistinct("trip_id")
        .alias("trip_id_unicos")

).show()

+---------------+----------------+-----------------+-------------------------+-----------------------+------------------+-----------+-------------+-----------+--------------+
|total_registros|invalid_duration|duration_mismatch|invalid_start_coordinates|invalid_end_coordinates|missing_birth_year|age_outlier|long_duration|valid_trips|trip_id_unicos|
+---------------+----------------+-----------------+-------------------------+-----------------------+------------------+-----------+-------------+-----------+--------------+
|        5676020|               0|              127|                        0|                     74|            649611|       2075|        13466|    5675946|       5676020|
+---------------+----------------+-----------------+-------------------------+-----------------------+------------------+-----------+-------------+-----------+--------------+



In [0]:
df_weather_clean = (
    df_weather
    .dropna(how="all")
    .filter(
        col("time").isNotNull()
    )
    .filter(
        (col("time") >= lit(FECHA_INICIO).cast("timestamp"))
        & (col("time") < lit(FECHA_FIN).cast("timestamp"))
    )
)

total_weather_bronze = df_weather.count()
total_weather_clean = df_weather_clean.count()

print(f"Weather Bronze        : {total_weather_bronze:,}")
print(f"Weather después filtro: {total_weather_clean:,}")
print(
    f"Registros excluidos   : "
    f"{total_weather_bronze - total_weather_clean:,}"
)

Weather Bronze        : 59,760
Weather después filtro: 4,368
Registros excluidos   : 55,392


In [0]:
weather_numeric_columns = [
    "temperature_2m_c",
    "precipitation_mm",
    "rain_mm",
    "cloudcover_pct",
    "cloudcover_low_pct",
    "cloudcover_mid_pct",
    "cloudcover_high_pct",
    "windspeed_10m_kmh",
    "winddirection_10m_deg"
]

df_weather_normalized = df_weather_clean

for columna in weather_numeric_columns:
    df_weather_normalized = (
        df_weather_normalized
        .withColumn(
            columna,
            when(
                isnan(col(columna)),
                lit(None).cast(DoubleType())
            ).otherwise(col(columna))
        )
    )

In [0]:
df_weather_dq = (
    df_weather_normalized

    # ============================================================
    # VALORES PRINCIPALES FALTANTES
    # ============================================================

    .withColumn(
        "dq_missing_core_weather",
        when(
            col("temperature_2m_c").isNull()
            | col("precipitation_mm").isNull()
            | col("rain_mm").isNull()
            | col("cloudcover_pct").isNull()
            | col("windspeed_10m_kmh").isNull(),
            lit(1)
        ).otherwise(lit(0))
    )

    # ============================================================
    # DIRECCIÓN DEL VIENTO FALTANTE
    # ============================================================

    .withColumn(
        "dq_missing_wind_direction",
        when(
            col("winddirection_10m_deg").isNull(),
            lit(1)
        ).otherwise(lit(0))
    )

    # ============================================================
    # PRECIPITACIÓN / LLUVIA
    # ============================================================

    .withColumn(
        "dq_invalid_precipitation",
        when(
            (col("precipitation_mm").isNotNull())
            & (col("precipitation_mm") < 0),
            lit(1)
        ).otherwise(lit(0))
    )

    .withColumn(
        "dq_invalid_rain",
        when(
            (col("rain_mm").isNotNull())
            & (col("rain_mm") < 0),
            lit(1)
        ).otherwise(lit(0))
    )

    # ============================================================
    # NUBOSIDAD
    # ============================================================

    .withColumn(
        "dq_invalid_cloudcover",
        when(
            (
                col("cloudcover_pct").isNotNull()
                & ~col("cloudcover_pct").between(0, 100)
            )
            |
            (
                col("cloudcover_low_pct").isNotNull()
                & ~col("cloudcover_low_pct").between(0, 100)
            )
            |
            (
                col("cloudcover_mid_pct").isNotNull()
                & ~col("cloudcover_mid_pct").between(0, 100)
            )
            |
            (
                col("cloudcover_high_pct").isNotNull()
                & ~col("cloudcover_high_pct").between(0, 100)
            ),
            lit(1)
        ).otherwise(lit(0))
    )

    # ============================================================
    # VIENTO
    # ============================================================

    .withColumn(
        "dq_invalid_windspeed",
        when(
            col("windspeed_10m_kmh").isNotNull()
            & (col("windspeed_10m_kmh") < 0),
            lit(1)
        ).otherwise(lit(0))
    )

    .withColumn(
        "dq_invalid_wind_direction",
        when(
            col("winddirection_10m_deg").isNotNull()
            & ~col("winddirection_10m_deg").between(0, 360),
            lit(1)
        ).otherwise(lit(0))
    )
)

In [0]:
df_weather_transform = (
    df_weather_dq

    # ============================================================
    # FLAG GLOBAL DE CALIDAD
    # ============================================================

    .withColumn(
        "dq_valid_weather",
        when(
            (col("dq_missing_core_weather") == 0)
            & (col("dq_invalid_precipitation") == 0)
            & (col("dq_invalid_rain") == 0)
            & (col("dq_invalid_cloudcover") == 0)
            & (col("dq_invalid_windspeed") == 0)
            & (col("dq_invalid_wind_direction") == 0),
            lit(1)
        ).otherwise(lit(0))
    )

    # ============================================================
    # VARIABLES TEMPORALES
    # ============================================================

    .withColumn(
        "weather_date",
        to_date(col("time"))
    )

    .withColumn(
        "weather_hour",
        hour(col("time"))
    )

    .withColumn(
        "weather_day_of_week",
        date_format(
            col("time"),
            "EEEE"
        )
    )

    .withColumn(
        "weather_month",
        month(col("time"))
    )

    .withColumn(
        "weather_year",
        year(col("time"))
    )

    .withColumn(
        "is_weekend",
        when(
            dayofweek(col("time")).isin(1, 7),
            lit(1)
        ).otherwise(lit(0))
    )

    # ============================================================
    # VARIABLES ANALÍTICAS
    # ============================================================

    .withColumn(
        "has_precipitation",
        when(
            col("precipitation_mm") > 0,
            lit(1)
        ).otherwise(lit(0))
    )

    .withColumn(
        "has_rain",
        when(
            col("rain_mm") > 0,
            lit(1)
        ).otherwise(lit(0))
    )

    .withColumn(
        "weather_condition",
        when(
            col("rain_mm") > 0,
            lit("RAIN")
        )
        .when(
            col("precipitation_mm") > 0,
            lit("PRECIPITATION")
        )
        .otherwise(lit("DRY"))
    )
)

In [0]:
df_weather_transform.select(
    count("*").alias("total_registros"),

    sum("dq_missing_core_weather")
        .alias("missing_core_weather"),

    sum("dq_missing_wind_direction")
        .alias("missing_wind_direction"),

    sum("dq_invalid_precipitation")
        .alias("invalid_precipitation"),

    sum("dq_invalid_rain")
        .alias("invalid_rain"),

    sum("dq_invalid_cloudcover")
        .alias("invalid_cloudcover"),

    sum("dq_invalid_windspeed")
        .alias("invalid_windspeed"),

    sum("dq_invalid_wind_direction")
        .alias("invalid_wind_direction"),

    sum(
        when(
            col("dq_valid_weather") == 1,
            lit(1)
        ).otherwise(lit(0))
    ).alias("valid_weather")

).show()

+---------------+--------------------+----------------------+---------------------+------------+------------------+-----------------+----------------------+-------------+
|total_registros|missing_core_weather|missing_wind_direction|invalid_precipitation|invalid_rain|invalid_cloudcover|invalid_windspeed|invalid_wind_direction|valid_weather|
+---------------+--------------------+----------------------+---------------------+------------+------------------+-----------------+----------------------+-------------+
|           4368|                   0|                     0|                    0|           0|                 0|                0|                     0|         4368|
+---------------+--------------------+----------------------+---------------------+------------+------------------+-----------------+----------------------+-------------+



In [0]:
df_weather_transform.select(
    min("time").alias("time_min"),
    max("time").alias("time_max"),

    min("weather_hour").alias("hour_min"),
    max("weather_hour").alias("hour_max"),

    countDistinct("weather_date")
        .alias("dias"),

    countDistinct("weather_month")
        .alias("meses"),

    countDistinct("weather_year")
        .alias("anios"),

    count("*")
        .alias("total_registros"),

    countDistinct("time")
        .alias("time_unicos")

).show()

+-------------------+-------------------+--------+--------+----+-----+-----+---------------+-----------+
|           time_min|           time_max|hour_min|hour_max|dias|meses|anios|total_registros|time_unicos|
+-------------------+-------------------+--------+--------+----+-----+-----+---------------+-----------+
|2016-01-01 00:00:00|2016-06-30 23:00:00|       0|      23| 182|    6|    1|           4368|       4368|
+-------------------+-------------------+--------+--------+----+-----+-----+---------------+-----------+



In [0]:
df_weather_silver = df_weather_transform.select(
    col("time"),

    col("temperature_2m_c"),
    col("precipitation_mm"),
    col("rain_mm"),

    col("cloudcover_pct"),
    col("cloudcover_low_pct"),
    col("cloudcover_mid_pct"),
    col("cloudcover_high_pct"),

    col("windspeed_10m_kmh"),
    col("winddirection_10m_deg"),

    col("_ingestion_timestamp"),
    col("_source_file"),
    col("_source_system"),
    col("_batch_id"),

    col("dq_missing_core_weather"),
    col("dq_missing_wind_direction"),
    col("dq_invalid_precipitation"),
    col("dq_invalid_rain"),
    col("dq_invalid_cloudcover"),
    col("dq_invalid_windspeed"),
    col("dq_invalid_wind_direction"),
    col("dq_valid_weather"),

    col("weather_date"),
    col("weather_hour"),
    col("weather_day_of_week"),
    col("weather_month"),
    col("weather_year"),
    col("is_weekend"),

    col("has_precipitation"),
    col("has_rain"),
    col("weather_condition")
)

In [0]:
tabla_weather_silver = (
    f"{catalogo}.{esquema_sink}.weather_hourly"
)

print("=== SCHEMA DATAFRAME WEATHER SILVER ===")
df_weather_silver.printSchema()

print("=== SCHEMA TABLA WEATHER SILVER ===")
spark.table(tabla_weather_silver).printSchema()

columnas_df = df_weather_silver.columns
columnas_tabla = spark.table(tabla_weather_silver).columns

print(f"Columnas DataFrame : {len(columnas_df)}")
print(f"Columnas tabla     : {len(columnas_tabla)}")

if columnas_df != columnas_tabla:
    raise Exception(
        "El orden o nombre de las columnas del DataFrame "
        "no coincide con la tabla Silver."
    )

print("Contrato de columnas Weather correcto.")

=== SCHEMA DATAFRAME WEATHER SILVER ===
root
 |-- time: timestamp (nullable = true)
 |-- temperature_2m_c: double (nullable = true)
 |-- precipitation_mm: double (nullable = true)
 |-- rain_mm: double (nullable = true)
 |-- cloudcover_pct: double (nullable = true)
 |-- cloudcover_low_pct: double (nullable = true)
 |-- cloudcover_mid_pct: double (nullable = true)
 |-- cloudcover_high_pct: double (nullable = true)
 |-- windspeed_10m_kmh: double (nullable = true)
 |-- winddirection_10m_deg: double (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_system: string (nullable = true)
 |-- _batch_id: string (nullable = true)
 |-- dq_missing_core_weather: integer (nullable = false)
 |-- dq_missing_wind_direction: integer (nullable = false)
 |-- dq_invalid_precipitation: integer (nullable = false)
 |-- dq_invalid_rain: integer (nullable = false)
 |-- dq_invalid_cloudcover: integer (nullable = false)
 |-- dq_invalid_wi

In [0]:
df_weather_silver.write.mode("overwrite")\
    .insertInto(tabla_weather_silver)

In [0]:
total_weather_bronze = df_weather.count()
total_weather_periodo = df_weather_clean.count()
total_weather_transform = df_weather_silver.count()
total_weather_silver = spark.table(
    tabla_weather_silver
).count()

print(f"Weather Bronze total    : {total_weather_bronze:,}")
print(f"Weather período proyecto: {total_weather_periodo:,}")
print(f"Weather Transform       : {total_weather_transform:,}")
print(f"Weather Silver          : {total_weather_silver:,}")

if total_weather_periodo != total_weather_transform:
    raise Exception(
        "La cantidad de registros del período y Transform no coincide."
    )

if total_weather_transform != total_weather_silver:
    raise Exception(
        "La cantidad de registros Transform y Silver no coincide."
    )

print("Transformación Weather finalizada correctamente.")

Weather Bronze total    : 59,760
Weather período proyecto: 4,368
Weather Transform       : 4,368
Weather Silver          : 4,368
Transformación Weather finalizada correctamente.


In [0]:
df_weather_silver_check = spark.table(
    tabla_weather_silver
)

df_weather_silver_check.select(
    count("*").alias("total_registros"),

    sum("dq_missing_core_weather")
        .alias("missing_core_weather"),

    sum("dq_missing_wind_direction")
        .alias("missing_wind_direction"),

    sum("dq_invalid_precipitation")
        .alias("invalid_precipitation"),

    sum("dq_invalid_rain")
        .alias("invalid_rain"),

    sum("dq_invalid_cloudcover")
        .alias("invalid_cloudcover"),

    sum("dq_invalid_windspeed")
        .alias("invalid_windspeed"),

    sum("dq_invalid_wind_direction")
        .alias("invalid_wind_direction"),

    sum(
        when(
            col("dq_valid_weather") == 1,
            lit(1)
        ).otherwise(lit(0))
    ).alias("valid_weather"),

    countDistinct("time")
        .alias("time_unicos")

).show()

+---------------+--------------------+----------------------+---------------------+------------+------------------+-----------------+----------------------+-------------+-----------+
|total_registros|missing_core_weather|missing_wind_direction|invalid_precipitation|invalid_rain|invalid_cloudcover|invalid_windspeed|invalid_wind_direction|valid_weather|time_unicos|
+---------------+--------------------+----------------------+---------------------+------------+------------------+-----------------+----------------------+-------------+-----------+
|           4368|                   0|                     0|                    0|           0|                 0|                0|                     0|         4368|       4368|
+---------------+--------------------+----------------------+---------------------+------------+------------------+-----------------+----------------------+-------------+-----------+



In [0]:
df_weather_silver_check.select(
    *[
        sum(
            when(
                isnan(col(columna)),
                lit(1)
            ).otherwise(lit(0))
        ).alias(f"{columna}_nan")
        for columna in weather_numeric_columns
    ]
).show()

+--------------------+--------------------+-----------+------------------+----------------------+----------------------+-----------------------+---------------------+-------------------------+
|temperature_2m_c_nan|precipitation_mm_nan|rain_mm_nan|cloudcover_pct_nan|cloudcover_low_pct_nan|cloudcover_mid_pct_nan|cloudcover_high_pct_nan|windspeed_10m_kmh_nan|winddirection_10m_deg_nan|
+--------------------+--------------------+-----------+------------------+----------------------+----------------------+-----------------------+---------------------+-------------------------+
|                   0|                   0|          0|                 0|                     0|                     0|                      0|                    0|                        0|
+--------------------+--------------------+-----------+------------------+----------------------+----------------------+-----------------------+---------------------+-------------------------+



In [0]:
df_taxi_zones_geojson_clean = (
    df_taxi_zones_geojson
    .dropna(how="all")
    .filter(
        col("objectid").isNotNull()
        & col("location_id").isNotNull()
    )
)

df_taxi_zones_csv_clean = (
    df_taxi_zones_csv
    .dropna(how="all")
    .filter(
        col("objectid").isNotNull()
        & col("location_id").isNotNull()
    )
)

total_geojson = df_taxi_zones_geojson.count()
total_geojson_clean = df_taxi_zones_geojson_clean.count()

total_csv = df_taxi_zones_csv.count()
total_csv_clean = df_taxi_zones_csv_clean.count()

print(f"GeoJSON Bronze        : {total_geojson:,}")
print(f"GeoJSON después filtro: {total_geojson_clean:,}")

print(f"CSV Bronze            : {total_csv:,}")
print(f"CSV después filtro    : {total_csv_clean:,}")

GeoJSON Bronze        : 263
GeoJSON después filtro: 263
CSV Bronze            : 263
CSV después filtro    : 263


In [0]:
df_taxi_zones_compare = (
    df_taxi_zones_geojson_clean.alias("g")
    .join(
        df_taxi_zones_csv_clean.alias("c"),
        col("g.objectid") == col("c.objectid"),
        "full"
    )
    .select(
        coalesce(
            col("g.objectid"),
            col("c.objectid")
        ).alias("objectid"),

        col("g.objectid").alias("geo_objectid"),
        col("c.objectid").alias("csv_objectid"),

        col("g.location_id").alias("geo_location_id"),
        col("c.location_id").alias("csv_location_id"),

        col("g.zone").alias("geo_zone"),
        col("c.zone").alias("csv_zone"),

        col("g.borough").alias("geo_borough"),
        col("c.borough").alias("csv_borough"),

        col("g.shape_area").alias("geo_shape_area"),
        col("c.shape_area").alias("csv_shape_area"),

        col("g.shape_leng").alias("geo_shape_leng"),
        col("c.shape_leng").alias("csv_shape_leng"),

        col("g.geometry").alias("geometry"),

        col("g._ingestion_timestamp")
            .alias("_ingestion_timestamp"),

        col("g._source_file")
            .alias("_source_file"),

        col("g._source_system")
            .alias("_source_system"),

        col("g._batch_id")
            .alias("_batch_id"),

        col("c._source_file")
            .alias("_csv_source_file")
    )
)

df_taxi_zones_compare = (
    df_taxi_zones_compare

    .withColumn(
        "dq_attribute_mismatch",
        when(
            col("geo_objectid").isNull()
            | col("csv_objectid").isNull()

            | (
                ~col("geo_location_id")
                .eqNullSafe(col("csv_location_id"))
            )

            | (
                ~trim(col("geo_zone"))
                .eqNullSafe(trim(col("csv_zone")))
            )

            | (
                ~trim(col("geo_borough"))
                .eqNullSafe(trim(col("csv_borough")))
            )

            | (
                ~round(col("geo_shape_area"), 6)
                .eqNullSafe(
                    round(col("csv_shape_area"), 6)
                )
            )

            | (
                ~round(col("geo_shape_leng"), 6)
                .eqNullSafe(
                    round(col("csv_shape_leng"), 6)
                )
            ),

            lit(1)
        ).otherwise(lit(0))
    )
)

In [0]:
df_taxi_zones_compare.select(
    count("*").alias("total_features"),

    sum(
        when(
            col("geo_objectid").isNull(),
            lit(1)
        ).otherwise(lit(0))
    ).alias("missing_geojson"),

    sum(
        when(
            col("csv_objectid").isNull(),
            lit(1)
        ).otherwise(lit(0))
    ).alias("missing_csv"),

    sum("dq_attribute_mismatch")
        .alias("attribute_mismatch")

).show()

+--------------+---------------+-----------+------------------+
|total_features|missing_geojson|missing_csv|attribute_mismatch|
+--------------+---------------+-----------+------------------+
|           263|              0|          0|                 0|
+--------------+---------------+-----------+------------------+



In [0]:
df_taxi_zone_features = (
    df_taxi_zones_compare

    .filter(
        col("geo_objectid").isNotNull()
    )

    .select(
        col("geo_objectid")
            .alias("objectid"),

        col("geo_location_id")
            .alias("location_id"),

        trim(col("geo_zone"))
            .alias("zone"),

        trim(col("geo_borough"))
            .alias("borough"),

        col("geo_shape_area")
            .alias("shape_area"),

        col("geo_shape_leng")
            .alias("shape_leng"),

        col("geometry"),

        col("dq_attribute_mismatch"),

        col("_ingestion_timestamp"),
        col("_source_file"),
        col("_source_system"),
        col("_batch_id"),

        col("_csv_source_file")
    )

    # ============================================================
    # UDF VISTA EN CLASE
    # ============================================================

    .withColumn(
        "borough_category",
        borough_categoria_udf(
            col("borough")
        )
    )

    # ============================================================
    # TIPO DE GEOMETRÍA
    # ============================================================

    .withColumn(
        "geometry_type",
        get_json_object(
            col("geometry"),
            "$.type"
        )
    )

    # ============================================================
    # REGLAS DQ
    # ============================================================

    .withColumn(
        "dq_missing_geometry",
        when(
            col("geometry").isNull()
            | (length(trim(col("geometry"))) == 0),
            lit(1)
        ).otherwise(lit(0))
    )

    .withColumn(
        "dq_invalid_geometry_type",
        when(
            col("geometry_type").isNull()
            | (col("geometry_type") != "MultiPolygon"),
            lit(1)
        ).otherwise(lit(0))
    )
)

df_taxi_zone_features = (
    df_taxi_zone_features
    .withColumn(
        "dq_valid_feature",
        when(
            col("objectid").isNotNull()
            & col("location_id").isNotNull()
            & col("zone").isNotNull()
            & col("borough").isNotNull()
            & (col("dq_attribute_mismatch") == 0)
            & (col("dq_missing_geometry") == 0)
            & (col("dq_invalid_geometry_type") == 0),
            lit(1)
        ).otherwise(lit(0))
    )
)

In [0]:
df_taxi_zone_features.select(
    count("*")
        .alias("total_features"),

    countDistinct("objectid")
        .alias("objectid_unicos"),

    countDistinct("location_id")
        .alias("location_id_unicos"),

    sum("dq_attribute_mismatch")
        .alias("attribute_mismatch"),

    sum("dq_missing_geometry")
        .alias("missing_geometry"),

    sum("dq_invalid_geometry_type")
        .alias("invalid_geometry_type"),

    sum(
        when(
            col("dq_valid_feature") == 1,
            lit(1)
        ).otherwise(lit(0))
    ).alias("valid_features")

).show()


df_taxi_zone_features.groupBy(
    "location_id"
).agg(
    count("*").alias("source_feature_count")
).filter(
    col("source_feature_count") > 1
).orderBy(
    "location_id"
).show()

+--------------+---------------+------------------+------------------+----------------+---------------------+--------------+
|total_features|objectid_unicos|location_id_unicos|attribute_mismatch|missing_geometry|invalid_geometry_type|valid_features|
+--------------+---------------+------------------+------------------+----------------+---------------------+--------------+
|           263|            263|               260|                 0|               0|                    0|           263|
+--------------+---------------+------------------+------------------+----------------+---------------------+--------------+

+-----------+--------------------+
|location_id|source_feature_count|
+-----------+--------------------+
|         56|                   2|
|        103|                   3|
+-----------+--------------------+



In [0]:
df_taxi_zones_transform = (
    df_taxi_zone_features

    .groupBy(
        "location_id",
        "zone",
        "borough",
        "borough_category"
    )

    .agg(
        count("*")
            .alias("source_feature_count"),

        min("dq_valid_feature")
            .alias("dq_valid_zone"),

        first(
            "_ingestion_timestamp",
            ignorenulls=True
        ).alias("_ingestion_timestamp"),

        first(
            "_source_file",
            ignorenulls=True
        ).alias("_source_file"),

        first(
            "_source_system",
            ignorenulls=True
        ).alias("_source_system"),

        first(
            "_batch_id",
            ignorenulls=True
        ).alias("_batch_id")
    )
)

In [0]:
df_taxi_zones_transform.select(
    count("*")
        .alias("total_zonas"),

    countDistinct("location_id")
        .alias("location_id_unicos"),

    sum(
        when(
            col("dq_valid_zone") == 1,
            lit(1)
        ).otherwise(lit(0))
    ).alias("zonas_validas"),

    sum(
        when(
            col("source_feature_count") > 1,
            lit(1)
        ).otherwise(lit(0))
    ).alias("zonas_multipart")

).show()

+-----------+------------------+-------------+---------------+
|total_zonas|location_id_unicos|zonas_validas|zonas_multipart|
+-----------+------------------+-------------+---------------+
|        260|               260|          260|              2|
+-----------+------------------+-------------+---------------+



In [0]:
df_taxi_zone_features.groupBy(
    "borough",
    "borough_category"
).agg(
    count("*").alias("total_features"),
    countDistinct("location_id").alias("total_zones")
).orderBy(
    "borough"
).show(
    truncate=False
)

+-------------+----------------+--------------+-----------+
|borough      |borough_category|total_features|total_zones|
+-------------+----------------+--------------+-----------+
|Bronx        |OUTER_BOROUGH   |43            |43         |
|Brooklyn     |OUTER_BOROUGH   |61            |61         |
|EWR          |OUTSIDE_NYC     |1             |1          |
|Manhattan    |MANHATTAN       |69            |67         |
|Queens       |OUTER_BOROUGH   |69            |68         |
|Staten Island|OUTER_BOROUGH   |20            |20         |
+-------------+----------------+--------------+-----------+



In [0]:
df_taxi_zone_features_silver = df_taxi_zone_features.select(
    col("objectid"),
    col("location_id"),
    col("zone"),
    col("borough"),

    col("shape_area"),
    col("shape_leng"),
    col("geometry"),

    col("dq_attribute_mismatch"),

    col("_ingestion_timestamp"),
    col("_source_file"),
    col("_source_system"),
    col("_batch_id"),
    col("_csv_source_file"),

    col("borough_category"),
    col("geometry_type"),

    col("dq_missing_geometry"),
    col("dq_invalid_geometry_type"),
    col("dq_valid_feature")
)

In [0]:
tabla_taxi_zone_features_silver = (
    f"{catalogo}.{esquema_sink}.taxi_zone_features"
)

print("=== SCHEMA DATAFRAME TAXI ZONE FEATURES ===")
df_taxi_zone_features_silver.printSchema()

print("=== SCHEMA TABLA TAXI ZONE FEATURES ===")
spark.table(
    tabla_taxi_zone_features_silver
).printSchema()

columnas_df = df_taxi_zone_features_silver.columns
columnas_tabla = spark.table(
    tabla_taxi_zone_features_silver
).columns

print(f"Columnas DataFrame : {len(columnas_df)}")
print(f"Columnas tabla     : {len(columnas_tabla)}")

if columnas_df != columnas_tabla:
    raise Exception(
        "El orden o nombre de las columnas de Taxi Zone Features "
        "no coincide con la tabla Silver."
    )

print("Contrato Taxi Zone Features correcto.")

=== SCHEMA DATAFRAME TAXI ZONE FEATURES ===
root
 |-- objectid: integer (nullable = true)
 |-- location_id: integer (nullable = true)
 |-- zone: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- shape_area: double (nullable = true)
 |-- shape_leng: double (nullable = true)
 |-- geometry: string (nullable = true)
 |-- dq_attribute_mismatch: integer (nullable = false)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_system: string (nullable = true)
 |-- _batch_id: string (nullable = true)
 |-- _csv_source_file: string (nullable = true)
 |-- borough_category: string (nullable = true)
 |-- geometry_type: string (nullable = true)
 |-- dq_missing_geometry: integer (nullable = false)
 |-- dq_invalid_geometry_type: integer (nullable = false)
 |-- dq_valid_feature: integer (nullable = false)

=== SCHEMA TABLA TAXI ZONE FEATURES ===
root
 |-- objectid: integer (nullable = true)
 |-- location_id: integer (nullable =

In [0]:
df_taxi_zone_features_silver.write.mode("overwrite")\
    .insertInto(tabla_taxi_zone_features_silver)

In [0]:
total_geojson_bronze = df_taxi_zones_geojson.count()
total_features_transform = df_taxi_zone_features_silver.count()
total_features_silver = spark.table(
    tabla_taxi_zone_features_silver
).count()

print(f"GeoJSON Bronze          : {total_geojson_bronze:,}")
print(f"Zone Features Transform : {total_features_transform:,}")
print(f"Zone Features Silver    : {total_features_silver:,}")

if total_geojson_bronze != total_features_transform:
    raise Exception(
        "Bronze GeoJSON y Features Transform no coinciden."
    )

if total_features_transform != total_features_silver:
    raise Exception(
        "Features Transform y Silver no coinciden."
    )

print("Taxi Zone Features finalizada correctamente.")

GeoJSON Bronze          : 263
Zone Features Transform : 263
Zone Features Silver    : 263
Taxi Zone Features finalizada correctamente.


In [0]:
df_taxi_zones_silver = df_taxi_zones_transform.select(
    col("location_id"),
    col("zone"),
    col("borough"),
    col("borough_category"),

    col("source_feature_count"),
    col("dq_valid_zone"),

    col("_ingestion_timestamp"),
    col("_source_file"),
    col("_source_system"),
    col("_batch_id")
)

In [0]:
tabla_taxi_zones_silver = (
    f"{catalogo}.{esquema_sink}.taxi_zones"
)

print("=== SCHEMA DATAFRAME TAXI ZONES ===")
df_taxi_zones_silver.printSchema()

print("=== SCHEMA TABLA TAXI ZONES ===")
spark.table(
    tabla_taxi_zones_silver
).printSchema()

columnas_df = df_taxi_zones_silver.columns
columnas_tabla = spark.table(
    tabla_taxi_zones_silver
).columns

print(f"Columnas DataFrame : {len(columnas_df)}")
print(f"Columnas tabla     : {len(columnas_tabla)}")

if columnas_df != columnas_tabla:
    raise Exception(
        "El orden o nombre de las columnas de Taxi Zones "
        "no coincide con la tabla Silver."
    )

print("Contrato Taxi Zones correcto.")

=== SCHEMA DATAFRAME TAXI ZONES ===
root
 |-- location_id: integer (nullable = true)
 |-- zone: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- borough_category: string (nullable = true)
 |-- source_feature_count: long (nullable = false)
 |-- dq_valid_zone: integer (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_system: string (nullable = true)
 |-- _batch_id: string (nullable = true)

=== SCHEMA TABLA TAXI ZONES ===
root
 |-- location_id: integer (nullable = true)
 |-- zone: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- borough_category: string (nullable = true)
 |-- source_feature_count: long (nullable = true)
 |-- dq_valid_zone: integer (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_system: string (nullable = true)
 |-- _batch_id: string (nullable = true)

Columnas DataFrame : 

In [0]:
df_taxi_zones_silver.write.mode("overwrite")\
    .insertInto(tabla_taxi_zones_silver)

In [0]:
total_features = df_taxi_zone_features_silver.count()
total_zonas_transform = df_taxi_zones_silver.count()
total_zonas_silver = spark.table(
    tabla_taxi_zones_silver
).count()

location_ids_features = (
    df_taxi_zone_features_silver
    .select("location_id")
    .distinct()
    .count()
)

print(f"Features físicas        : {total_features:,}")
print(f"Location ID distintos   : {location_ids_features:,}")
print(f"Zonas Transform         : {total_zonas_transform:,}")
print(f"Zonas Silver            : {total_zonas_silver:,}")

if location_ids_features != total_zonas_transform:
    raise Exception(
        "Los Location ID distintos no coinciden "
        "con la dimensión lógica."
    )

if total_zonas_transform != total_zonas_silver:
    raise Exception(
        "Taxi Zones Transform y Silver no coinciden."
    )

print("Dimensión Taxi Zones finalizada correctamente.")

Features físicas        : 263
Location ID distintos   : 260
Zonas Transform         : 260
Zonas Silver            : 260
Dimensión Taxi Zones finalizada correctamente.


In [0]:
df_zone_features_check = spark.table(
    tabla_taxi_zone_features_silver
)

df_zones_check = spark.table(
    tabla_taxi_zones_silver
)

print("=== TAXI ZONE FEATURES SILVER ===")

df_zone_features_check.select(
    count("*").alias("total_features"),
    countDistinct("objectid").alias("objectid_unicos"),
    countDistinct("location_id").alias("location_id_unicos"),

    sum("dq_attribute_mismatch")
        .alias("attribute_mismatch"),

    sum("dq_missing_geometry")
        .alias("missing_geometry"),

    sum("dq_invalid_geometry_type")
        .alias("invalid_geometry_type"),

    sum(
        when(
            col("dq_valid_feature") == 1,
            lit(1)
        ).otherwise(lit(0))
    ).alias("valid_features")
).show()


print("=== TAXI ZONES SILVER ===")

df_zones_check.select(
    count("*").alias("total_zonas"),

    countDistinct("location_id")
        .alias("location_id_unicos"),

    sum(
        when(
            col("dq_valid_zone") == 1,
            lit(1)
        ).otherwise(lit(0))
    ).alias("zonas_validas"),

    sum(
        when(
            col("source_feature_count") > 1,
            lit(1)
        ).otherwise(lit(0))
    ).alias("zonas_multipart")
).show()

=== TAXI ZONE FEATURES SILVER ===
+--------------+---------------+------------------+------------------+----------------+---------------------+--------------+
|total_features|objectid_unicos|location_id_unicos|attribute_mismatch|missing_geometry|invalid_geometry_type|valid_features|
+--------------+---------------+------------------+------------------+----------------+---------------------+--------------+
|           263|            263|               260|                 0|               0|                    0|           263|
+--------------+---------------+------------------+------------------+----------------+---------------------+--------------+

=== TAXI ZONES SILVER ===
+-----------+------------------+-------------+---------------+
|total_zonas|location_id_unicos|zonas_validas|zonas_multipart|
+-----------+------------------+-------------+---------------+
|        260|               260|          260|              2|
+-----------+------------------+-------------+---------------+

In [0]:
# ============================================================
# VERIFICAR CAPACIDADES GEOESPACIALES
# ============================================================

print("=== ENTORNO DATABRICKS ===")

print(
    "Spark Version:",
    spark.version
)

runtime_version = spark.conf.get(
    "spark.databricks.clusterUsageTags.sparkVersion",
    "NO_DISPONIBLE"
)

print(
    "Databricks Runtime:",
    runtime_version
)

print("\n=== PRUEBA FUNCIONES GEOESPACIALES NATIVAS ===")

try:
    from pyspark.databricks.sql import functions as dbf

    print("Import pyspark.databricks.sql.functions: OK")

    df_geo_test = spark.createDataFrame(
        [
            (
                '{"type":"Polygon","coordinates":[[[0,0],[10,0],[0,10],[0,0]]]}',
                1.0,
                1.0
            )
        ],
        [
            "geojson",
            "longitude",
            "latitude"
        ]
    )

    df_geo_test = (
        df_geo_test

        .withColumn(
            "polygon_geom",
            dbf.st_geomfromgeojson(
                col("geojson")
            )
        )

        .withColumn(
            "point_geom",
            dbf.st_point(
                col("longitude"),
                col("latitude"),
                4326
            )
        )

        .withColumn(
            "point_inside",
            dbf.st_contains(
                col("polygon_geom"),
                col("point_geom")
            )
        )
    )

    df_geo_test.select(
        "longitude",
        "latitude",
        "point_inside"
    ).show(
        truncate=False
    )

    print(
        "Funciones geoespaciales nativas disponibles: SI"
    )

except Exception as e:

    print(
        "Funciones geoespaciales nativas disponibles: NO"
    )

    print(
        "Detalle:",
        str(e)
    )

=== ENTORNO DATABRICKS ===
Spark Version: 4.0.0
Databricks Runtime: 17.3.x-aarch64-photon-scala2.13

=== PRUEBA FUNCIONES GEOESPACIALES NATIVAS ===
Import pyspark.databricks.sql.functions: OK
+---------+--------+------------+
|longitude|latitude|point_inside|
+---------+--------+------------+
|1.0      |1.0     |true        |
+---------+--------+------------+

Funciones geoespaciales nativas disponibles: SI


In [0]:
# ============================================================
# PREPARAR GEOMETRÍAS TAXI ZONES
# ============================================================

df_taxi_zone_features_geo = (
    spark.table(
        tabla_taxi_zone_features_silver
    )

    .filter(
        col("dq_valid_feature") == 1
    )

    .withColumn(
        "zone_geometry",
        dbf.st_geomfromgeojson(
            col("geometry")
        )
    )

    .withColumn(
        "zone_geometry_type",
        dbf.st_geometrytype(
            col("zone_geometry")
        )
    )

    .withColumn(
        "zone_srid",
        dbf.st_srid(
            col("zone_geometry")
        )
    )
)

In [0]:
# ============================================================
# VALIDAR GEOMETRÍAS TAXI ZONES
# ============================================================

df_taxi_zone_features_geo.select(
    count("*").alias("total_features"),

    countDistinct("objectid")
        .alias("objectid_unicos"),

    countDistinct("location_id")
        .alias("location_id_unicos"),

    sum(
        when(
            col("zone_geometry").isNull(),
            lit(1)
        ).otherwise(lit(0))
    ).alias("geometry_null"),

    countDistinct("zone_geometry_type")
        .alias("geometry_types"),

    countDistinct("zone_srid")
        .alias("srid_distintos")

).show()


print("=== TIPOS DE GEOMETRÍA ===")

df_taxi_zone_features_geo.groupBy(
    "zone_geometry_type"
).agg(
    count("*").alias("total")
).orderBy(
    "zone_geometry_type"
).show(
    truncate=False
)


print("=== SRID ===")

df_taxi_zone_features_geo.groupBy(
    "zone_srid"
).agg(
    count("*").alias("total")
).orderBy(
    "zone_srid"
).show()

+--------------+---------------+------------------+-------------+--------------+--------------+
|total_features|objectid_unicos|location_id_unicos|geometry_null|geometry_types|srid_distintos|
+--------------+---------------+------------------+-------------+--------------+--------------+
|           263|            263|               260|            0|             1|             1|
+--------------+---------------+------------------+-------------+--------------+--------------+

=== TIPOS DE GEOMETRÍA ===
+------------------+-----+
|zone_geometry_type|total|
+------------------+-----+
|ST_MultiPolygon   |263  |
+------------------+-----+

=== SRID ===
+---------+-----+
|zone_srid|total|
+---------+-----+
|     4326|  263|
+---------+-----+



In [0]:
df_taxi_geo = (
    spark.table(
        tabla_taxi_silver
    )

    # ============================================================
    # PUNTO PICKUP
    # ============================================================

    .withColumn(
        "pickup_point",
        dbf.st_point(
            col("pickup_longitude"),
            col("pickup_latitude"),
            4326
        )
    )

    # ============================================================
    # PUNTO DROPOFF
    # ============================================================

    .withColumn(
        "dropoff_point",
        dbf.st_point(
            col("dropoff_longitude"),
            col("dropoff_latitude"),
            4326
        )
    )

    # ============================================================
    # TIPO DE GEOMETRÍA
    # ============================================================

    .withColumn(
        "pickup_geometry_type",
        dbf.st_geometrytype(
            col("pickup_point")
        )
    )

    .withColumn(
        "dropoff_geometry_type",
        dbf.st_geometrytype(
            col("dropoff_point")
        )
    )

    # ============================================================
    # SRID
    # ============================================================

    .withColumn(
        "pickup_srid",
        dbf.st_srid(
            col("pickup_point")
        )
    )

    .withColumn(
        "dropoff_srid",
        dbf.st_srid(
            col("dropoff_point")
        )
    )
)

In [0]:
# ============================================================
# VALIDAR PUNTOS GEOESPACIALES TAXI
# ============================================================

df_taxi_geo.select(

    count("*")
        .alias("total_taxi"),

    sum(
        when(
            col("pickup_point").isNull(),
            lit(1)
        ).otherwise(lit(0))
    ).alias("pickup_point_null"),

    sum(
        when(
            col("dropoff_point").isNull(),
            lit(1)
        ).otherwise(lit(0))
    ).alias("dropoff_point_null"),

    countDistinct(
        "pickup_geometry_type"
    ).alias("pickup_geometry_types"),

    countDistinct(
        "dropoff_geometry_type"
    ).alias("dropoff_geometry_types"),

    countDistinct(
        "pickup_srid"
    ).alias("pickup_srid_distintos"),

    countDistinct(
        "dropoff_srid"
    ).alias("dropoff_srid_distintos")

).show()


print("=== TIPOS DE GEOMETRÍA TAXI ===")

df_taxi_geo.groupBy(
    "pickup_geometry_type",
    "dropoff_geometry_type"
).agg(
    count("*").alias("total")
).show(
    truncate=False
)


print("=== SRID TAXI ===")

df_taxi_geo.groupBy(
    "pickup_srid",
    "dropoff_srid"
).agg(
    count("*").alias("total")
).show()

+----------+-----------------+------------------+---------------------+----------------------+---------------------+----------------------+
|total_taxi|pickup_point_null|dropoff_point_null|pickup_geometry_types|dropoff_geometry_types|pickup_srid_distintos|dropoff_srid_distintos|
+----------+-----------------+------------------+---------------------+----------------------+---------------------+----------------------+
|   1458644|                0|                 0|                    1|                     1|                    1|                     1|
+----------+-----------------+------------------+---------------------+----------------------+---------------------+----------------------+

=== TIPOS DE GEOMETRÍA TAXI ===
+--------------------+---------------------+-------+
|pickup_geometry_type|dropoff_geometry_type|total  |
+--------------------+---------------------+-------+
|ST_Point            |ST_Point             |1458644|
+--------------------+---------------------+-------+

=

In [0]:
# SPATIAL JOIN PICKUP -> TAXI ZONE

df_zones_pickup = (
    df_taxi_zone_features_geo
    .select(
        col("objectid")
            .alias("pickup_zone_objectid"),

        col("location_id")
            .alias("pickup_location_id"),

        col("zone")
            .alias("pickup_zone"),

        col("borough")
            .alias("pickup_borough"),

        col("borough_category")
            .alias("pickup_borough_category"),

        col("zone_geometry")
    )
)


df_taxi_pickup_zone = (
    df_taxi_geo.alias("t")

    .join(
        F.broadcast(
            df_zones_pickup.alias("z")
        ),

        dbf.st_covers(
            col("z.zone_geometry"),
            col("t.pickup_point")
        ),

        "left"
    )

    .select(
        col("t.*"),

        col("z.pickup_zone_objectid"),
        col("z.pickup_location_id"),
        col("z.pickup_zone"),
        col("z.pickup_borough"),
        col("z.pickup_borough_category")
    )
)

In [0]:
#  VALIDAR SPATIAL JOIN PICKUP

total_taxi_original = df_taxi_geo.count()

total_filas_join = (
    df_taxi_pickup_zone
    .count()
)

total_ids_join = (
    df_taxi_pickup_zone
    .select("id")
    .distinct()
    .count()
)

sin_pickup_zone = (
    df_taxi_pickup_zone
    .filter(
        col("pickup_zone_objectid").isNull()
    )
    .count()
)


df_pickup_matches = (
    df_taxi_pickup_zone

    .groupBy("id")

    .agg(
        count(
            "pickup_zone_objectid"
        ).alias("zone_matches")
    )
)


sin_match = (
    df_pickup_matches
    .filter(
        col("zone_matches") == 0
    )
    .count()
)

un_match = (
    df_pickup_matches
    .filter(
        col("zone_matches") == 1
    )
    .count()
)

multiples_matches = (
    df_pickup_matches
    .filter(
        col("zone_matches") > 1
    )
    .count()
)


print("=== VALIDACIÓN SPATIAL JOIN PICKUP ===")

print(
    f"Taxi original          : "
    f"{total_taxi_original:,}"
)

print(
    f"Filas después join     : "
    f"{total_filas_join:,}"
)

print(
    f"IDs distintos después  : "
    f"{total_ids_join:,}"
)

print(
    f"Sin Pickup Zone        : "
    f"{sin_pickup_zone:,}"
)

print()
print("=== CANTIDAD DE MATCHES POR VIAJE ===")

print(
    f"Sin match              : "
    f"{sin_match:,}"
)

print(
    f"Con 1 match            : "
    f"{un_match:,}"
)

print(
    f"Con >1 match           : "
    f"{multiples_matches:,}"
)

=== VALIDACIÓN SPATIAL JOIN PICKUP ===
Taxi original          : 1,458,644
Filas después join     : 1,458,644
IDs distintos después  : 1,458,644
Sin Pickup Zone        : 1,121

=== CANTIDAD DE MATCHES POR VIAJE ===
Sin match              : 1,121
Con 1 match            : 1,457,523
Con >1 match           : 0


In [0]:
df_zones_dropoff = (
    df_taxi_zone_features_geo
    .select(
        col("objectid")
            .alias("dropoff_zone_objectid"),

        col("location_id")
            .alias("dropoff_location_id"),

        col("zone")
            .alias("dropoff_zone"),

        col("borough")
            .alias("dropoff_borough"),

        col("borough_category")
            .alias("dropoff_borough_category"),

        col("zone_geometry")
    )
)


df_taxi_zones_enriched = (
    df_taxi_pickup_zone.alias("t")

    .join(
        F.broadcast(
            df_zones_dropoff.alias("z")
        ),

        dbf.st_covers(
            col("z.zone_geometry"),
            col("t.dropoff_point")
        ),

        "left"
    )

    .select(
        col("t.*"),

        col("z.dropoff_zone_objectid"),
        col("z.dropoff_location_id"),
        col("z.dropoff_zone"),
        col("z.dropoff_borough"),
        col("z.dropoff_borough_category")
    )
)

In [0]:
total_taxi_original = (
    df_taxi_geo
    .count()
)

total_filas_join = (
    df_taxi_zones_enriched
    .count()
)

total_ids_join = (
    df_taxi_zones_enriched
    .select("id")
    .distinct()
    .count()
)

sin_dropoff_zone = (
    df_taxi_zones_enriched
    .filter(
        col("dropoff_zone_objectid").isNull()
    )
    .count()
)


df_dropoff_matches = (
    df_taxi_zones_enriched

    .groupBy("id")

    .agg(
        count(
            "dropoff_zone_objectid"
        ).alias("zone_matches")
    )
)


sin_match = (
    df_dropoff_matches
    .filter(
        col("zone_matches") == 0
    )
    .count()
)

un_match = (
    df_dropoff_matches
    .filter(
        col("zone_matches") == 1
    )
    .count()
)

multiples_matches = (
    df_dropoff_matches
    .filter(
        col("zone_matches") > 1
    )
    .count()
)


print("=== VALIDACIÓN SPATIAL JOIN DROPOFF ===")

print(
    f"Taxi original          : "
    f"{total_taxi_original:,}"
)

print(
    f"Filas después join     : "
    f"{total_filas_join:,}"
)

print(
    f"IDs distintos después  : "
    f"{total_ids_join:,}"
)

print(
    f"Sin Dropoff Zone       : "
    f"{sin_dropoff_zone:,}"
)

print()
print("=== CANTIDAD DE MATCHES POR VIAJE ===")

print(
    f"Sin match              : "
    f"{sin_match:,}"
)

print(
    f"Con 1 match            : "
    f"{un_match:,}"
)

print(
    f"Con >1 match           : "
    f"{multiples_matches:,}"
)

=== VALIDACIÓN SPATIAL JOIN DROPOFF ===
Taxi original          : 1,458,644
Filas después join     : 1,458,644
IDs distintos después  : 1,458,644
Sin Dropoff Zone       : 3,744

=== CANTIDAD DE MATCHES POR VIAJE ===
Sin match              : 3,744
Con 1 match            : 1,454,900
Con >1 match           : 0


In [0]:
# DQ GEOESPACIAL TAXI

df_taxi_geo_transform = (
    df_taxi_zones_enriched

    # ============================================================
    # PICKUP SIN TAXI ZONE
    # ============================================================

    .withColumn(
        "dq_missing_pickup_zone",
        when(
            col("pickup_location_id").isNull(),
            lit(1)
        ).otherwise(lit(0))
    )

    # ============================================================
    # DROPOFF SIN TAXI ZONE
    # ============================================================

    .withColumn(
        "dq_missing_dropoff_zone",
        when(
            col("dropoff_location_id").isNull(),
            lit(1)
        ).otherwise(lit(0))
    )

    # ============================================================
    # VALIDEZ GEOESPACIAL
    # ============================================================

    .withColumn(
        "dq_valid_spatial",
        when(
            col("pickup_location_id").isNotNull()
            & col("dropoff_location_id").isNotNull(),
            lit(1)
        ).otherwise(lit(0))
    )
)

In [0]:
# VALIDAR DQ GEOESPACIAL TAXI 

df_taxi_geo_transform.select(
    count("*")
        .alias("total_registros"),

    sum("dq_missing_pickup_zone")
        .alias("missing_pickup_zone"),

    sum("dq_missing_dropoff_zone")
        .alias("missing_dropoff_zone"),

    sum(
        when(
            col("dq_valid_spatial") == 1,
            lit(1)
        ).otherwise(lit(0))
    ).alias("valid_spatial"),

    sum(
        when(
            (col("dq_missing_pickup_zone") == 0)
            & (col("dq_missing_dropoff_zone") == 0),
            lit(1)
        ).otherwise(lit(0))
    ).alias("pickup_ok_dropoff_ok"),

    sum(
        when(
            (col("dq_missing_pickup_zone") == 1)
            & (col("dq_missing_dropoff_zone") == 0),
            lit(1)
        ).otherwise(lit(0))
    ).alias("pickup_missing_only"),

    sum(
        when(
            (col("dq_missing_pickup_zone") == 0)
            & (col("dq_missing_dropoff_zone") == 1),
            lit(1)
        ).otherwise(lit(0))
    ).alias("dropoff_missing_only"),

    sum(
        when(
            (col("dq_missing_pickup_zone") == 1)
            & (col("dq_missing_dropoff_zone") == 1),
            lit(1)
        ).otherwise(lit(0))
    ).alias("both_missing")

).show()

+---------------+-------------------+--------------------+-------------+--------------------+-------------------+--------------------+------------+
|total_registros|missing_pickup_zone|missing_dropoff_zone|valid_spatial|pickup_ok_dropoff_ok|pickup_missing_only|dropoff_missing_only|both_missing|
+---------------+-------------------+--------------------+-------------+--------------------+-------------------+--------------------+------------+
|        1458644|               1121|                3744|      1454641|             1454641|                259|                2882|         862|
+---------------+-------------------+--------------------+-------------+--------------------+-------------------+--------------------+------------+



In [0]:
# PREPARAR TAXI ENRIQUECIDO PARA SILVER 

df_taxi_enriched_silver = (
    df_taxi_geo_transform

    .select(
        # =====================================================
        # DATOS ORIGINALES TAXI
        # =====================================================

        col("id"),
        col("vendor_id"),
        col("pickup_datetime"),
        col("dropoff_datetime"),
        col("passenger_count"),

        col("pickup_longitude"),
        col("pickup_latitude"),
        col("dropoff_longitude"),
        col("dropoff_latitude"),

        col("store_and_fwd_flag"),
        col("trip_duration"),

        # =====================================================
        # METADATA
        # =====================================================

        col("_ingestion_timestamp"),
        col("_source_file"),
        col("_source_system"),
        col("_batch_id"),

        # =====================================================
        # DQ ORIGINAL
        # =====================================================

        col("dq_invalid_duration"),
        col("dq_invalid_passenger_count"),
        col("dq_invalid_pickup_coordinates"),
        col("dq_invalid_dropoff_coordinates"),
        col("dq_long_duration"),
        col("dq_valid_trip"),

        # =====================================================
        # VARIABLES DERIVADAS
        # =====================================================

        col("duration_minutes"),
        col("duration_hours"),
        col("pickup_date"),
        col("pickup_hour"),
        col("pickup_day_of_week"),
        col("pickup_month"),
        col("pickup_year"),
        col("is_weekend"),

        # =====================================================
        # PICKUP TAXI ZONE
        # =====================================================

        col("pickup_zone_objectid"),
        col("pickup_location_id"),
        col("pickup_zone"),
        col("pickup_borough"),
        col("pickup_borough_category"),

        # =====================================================
        # DROPOFF TAXI ZONE
        # =====================================================

        col("dropoff_zone_objectid"),
        col("dropoff_location_id"),
        col("dropoff_zone"),
        col("dropoff_borough"),
        col("dropoff_borough_category"),

        # =====================================================
        # DQ GEOESPACIAL
        # =====================================================

        col("dq_missing_pickup_zone"),
        col("dq_missing_dropoff_zone"),
        col("dq_valid_spatial")
    )
)

In [0]:
# VALIDAR TAXI ENRIQUECIDO 

print(
    f"Columnas Taxi enriquecido: "
    f"{len(df_taxi_enriched_silver.columns)}"
)

print(
    f"Registros Taxi enriquecido: "
    f"{df_taxi_enriched_silver.count():,}"
)


df_taxi_enriched_silver.select(
    count("*")
        .alias("total_registros"),

    countDistinct("id")
        .alias("ids_unicos"),

    sum("dq_missing_pickup_zone")
        .alias("missing_pickup_zone"),

    sum("dq_missing_dropoff_zone")
        .alias("missing_dropoff_zone"),

    sum(
        when(
            col("dq_valid_spatial") == 1,
            lit(1)
        ).otherwise(lit(0))
    ).alias("valid_spatial")
).show()

Columnas Taxi enriquecido: 42
Registros Taxi enriquecido: 1,458,644
+---------------+----------+-------------------+--------------------+-------------+
|total_registros|ids_unicos|missing_pickup_zone|missing_dropoff_zone|valid_spatial|
+---------------+----------+-------------------+--------------------+-------------+
|        1458644|   1458644|               1121|                3744|      1454641|
+---------------+----------+-------------------+--------------------+-------------+



In [0]:
tabla_taxi_enriched_silver = (
    f"{catalogo}.{esquema_sink}.taxi_trips_enriched"
)


print("=== SCHEMA DATAFRAME TAXI TRIPS ENRICHED ===")

df_taxi_enriched_silver.printSchema()


print("=== SCHEMA TABLA TAXI TRIPS ENRICHED ===")

spark.table(
    tabla_taxi_enriched_silver
).printSchema()


# ============================================================
# VALIDAR NOMBRE Y ORDEN DE COLUMNAS
# ============================================================

columnas_df = (
    df_taxi_enriched_silver
    .columns
)

columnas_tabla = (
    spark.table(
        tabla_taxi_enriched_silver
    )
    .columns
)


print(
    f"Columnas DataFrame : {len(columnas_df)}"
)

print(
    f"Columnas tabla     : {len(columnas_tabla)}"
)


if columnas_df != columnas_tabla:
    raise Exception(
        "El orden o nombre de las columnas de "
        "Taxi Trips Enriched no coincide con la tabla Silver."
    )


# ============================================================
# VALIDAR TIPOS DE DATOS
# ============================================================

schema_df = [
    (
        field.name,
        field.dataType.simpleString()
    )
    for field
    in df_taxi_enriched_silver.schema.fields
]


schema_tabla = [
    (
        field.name,
        field.dataType.simpleString()
    )
    for field
    in spark.table(
        tabla_taxi_enriched_silver
    ).schema.fields
]


if schema_df != schema_tabla:
    print(
        "=== SCHEMA DATAFRAME ==="
    )

    print(
        schema_df
    )

    print(
        "=== SCHEMA TABLA ==="
    )

    print(
        schema_tabla
    )

    raise Exception(
        "Los tipos de datos de Taxi Trips Enriched "
        "no coinciden con la tabla Silver."
    )


print(
    "Contrato Taxi Trips Enriched correcto."
)

=== SCHEMA DATAFRAME TAXI TRIPS ENRICHED ===
root
 |-- id: string (nullable = true)
 |-- vendor_id: integer (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- trip_duration: long (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_system: string (nullable = true)
 |-- _batch_id: string (nullable = true)
 |-- dq_invalid_duration: integer (nullable = true)
 |-- dq_invalid_passenger_count: integer (nullable = true)
 |-- dq_invalid_pickup_coordinates: integer (nullable = true)
 |-- dq_invalid_dropoff_coordinates: integer (nullable = true)
 |-- dq_long_durat

In [0]:
df_taxi_enriched_silver.write\
    .mode("overwrite")\
    .insertInto(
        tabla_taxi_enriched_silver
    )

In [0]:
total_taxi_silver = (
    spark.table(
        tabla_taxi_silver
    )
    .count()
)


total_taxi_enriched_transform = (
    df_taxi_enriched_silver
    .count()
)


total_taxi_enriched_silver = (
    spark.table(
        tabla_taxi_enriched_silver
    )
    .count()
)


ids_taxi_enriched_silver = (
    spark.table(
        tabla_taxi_enriched_silver
    )
    .select("id")
    .distinct()
    .count()
)


print("=== RECONCILIACIÓN TAXI ENRICHED ===")

print(
    f"Taxi Silver original      : "
    f"{total_taxi_silver:,}"
)

print(
    f"Taxi Enriched Transform   : "
    f"{total_taxi_enriched_transform:,}"
)

print(
    f"Taxi Enriched Silver      : "
    f"{total_taxi_enriched_silver:,}"
)

print(
    f"IDs únicos Enriched Silver: "
    f"{ids_taxi_enriched_silver:,}"
)


# ============================================================
# VALIDACIONES
# ============================================================

if total_taxi_silver != total_taxi_enriched_transform:
    raise Exception(
        "Taxi Silver y Taxi Enriched Transform "
        "no tienen la misma cantidad de registros."
    )


if (
    total_taxi_enriched_transform
    != total_taxi_enriched_silver
):
    raise Exception(
        "Taxi Enriched Transform y Taxi Enriched Silver "
        "no tienen la misma cantidad de registros."
    )


if (
    total_taxi_enriched_silver
    != ids_taxi_enriched_silver
):
    raise Exception(
        "Taxi Trips Enriched contiene IDs duplicados."
    )


print(
    "Taxi Trips Enriched finalizada correctamente."
)

=== RECONCILIACIÓN TAXI ENRICHED ===
Taxi Silver original      : 1,458,644
Taxi Enriched Transform   : 1,458,644
Taxi Enriched Silver      : 1,458,644
IDs únicos Enriched Silver: 1,458,644
Taxi Trips Enriched finalizada correctamente.


In [0]:
df_taxi_enriched_check = (
    spark.table(
        tabla_taxi_enriched_silver
    )
)


df_taxi_enriched_check.select(

    # ========================================================
    # VOLUMEN
    # ========================================================

    count("*")
        .alias("total_registros"),

    countDistinct("id")
        .alias("ids_unicos"),

    # ========================================================
    # DQ GEOESPACIAL
    # ========================================================

    sum("dq_missing_pickup_zone")
        .alias("missing_pickup_zone"),

    sum("dq_missing_dropoff_zone")
        .alias("missing_dropoff_zone"),

    sum(
        when(
            col("dq_valid_spatial") == 1,
            lit(1)
        ).otherwise(lit(0))
    ).alias("valid_spatial"),

    # ========================================================
    # COMBINACIONES
    # ========================================================

    sum(
        when(
            (col("dq_missing_pickup_zone") == 1)
            & (col("dq_missing_dropoff_zone") == 0),
            lit(1)
        ).otherwise(lit(0))
    ).alias("pickup_missing_only"),

    sum(
        when(
            (col("dq_missing_pickup_zone") == 0)
            & (col("dq_missing_dropoff_zone") == 1),
            lit(1)
        ).otherwise(lit(0))
    ).alias("dropoff_missing_only"),

    sum(
        when(
            (col("dq_missing_pickup_zone") == 1)
            & (col("dq_missing_dropoff_zone") == 1),
            lit(1)
        ).otherwise(lit(0))
    ).alias("both_missing"),

    # ========================================================
    # DQ ORIGINAL TAXI
    # ========================================================

    sum("dq_invalid_duration")
        .alias("invalid_duration"),

    sum("dq_invalid_passenger_count")
        .alias("invalid_passenger_count"),

    sum("dq_invalid_pickup_coordinates")
        .alias("invalid_pickup_coordinates"),

    sum("dq_invalid_dropoff_coordinates")
        .alias("invalid_dropoff_coordinates"),

    sum("dq_long_duration")
        .alias("long_duration"),

    sum(
        when(
            col("dq_valid_trip") == 1,
            lit(1)
        ).otherwise(lit(0))
    ).alias("valid_trip")

).show(
    truncate=False
)

+---------------+----------+-------------------+--------------------+-------------+-------------------+--------------------+------------+----------------+-----------------------+--------------------------+---------------------------+-------------+----------+
|total_registros|ids_unicos|missing_pickup_zone|missing_dropoff_zone|valid_spatial|pickup_missing_only|dropoff_missing_only|both_missing|invalid_duration|invalid_passenger_count|invalid_pickup_coordinates|invalid_dropoff_coordinates|long_duration|valid_trip|
+---------------+----------+-------------------+--------------------+-------------+-------------------+--------------------+------------+----------------+-----------------------+--------------------------+---------------------------+-------------+----------+
|1458644        |1458644   |1121               |3744                |1454641      |259                |2882                |862         |0               |60                     |0                         |0                 

In [0]:
df_citibike_geo = (
    spark.table(
        tabla_citibike_silver
    )

    # ============================================================
    # PUNTO START
    # Solo si las coordenadas son válidas
    # ============================================================

    .withColumn(
        "start_point",
        when(
            col("dq_invalid_start_coordinates") == 0,

            dbf.st_point(
                col("start_station_longitude"),
                col("start_station_latitude"),
                4326
            )
        )
    )

    # ============================================================
    # PUNTO END
    # Solo si las coordenadas son válidas
    # ============================================================

    .withColumn(
        "end_point",
        when(
            col("dq_invalid_end_coordinates") == 0,

            dbf.st_point(
                col("end_station_longitude"),
                col("end_station_latitude"),
                4326
            )
        )
    )

    # ============================================================
    # TIPOS DE GEOMETRÍA
    # ============================================================

    .withColumn(
        "start_geometry_type",
        dbf.st_geometrytype(
            col("start_point")
        )
    )

    .withColumn(
        "end_geometry_type",
        dbf.st_geometrytype(
            col("end_point")
        )
    )

    # ============================================================
    # SRID
    # ============================================================

    .withColumn(
        "start_srid",
        dbf.st_srid(
            col("start_point")
        )
    )

    .withColumn(
        "end_srid",
        dbf.st_srid(
            col("end_point")
        )
    )
)

In [0]:
df_citibike_geo.select(

    count("*")
        .alias("total_citibike"),

    sum(
        when(
            col("start_point").isNull(),
            lit(1)
        ).otherwise(lit(0))
    ).alias("start_point_null"),

    sum(
        when(
            col("end_point").isNull(),
            lit(1)
        ).otherwise(lit(0))
    ).alias("end_point_null"),

    countDistinct(
        "start_geometry_type"
    ).alias("start_geometry_types"),

    countDistinct(
        "end_geometry_type"
    ).alias("end_geometry_types"),

    countDistinct(
        "start_srid"
    ).alias("start_srid_distintos"),

    countDistinct(
        "end_srid"
    ).alias("end_srid_distintos")

).show()


print("=== TIPOS DE GEOMETRÍA CITIBIKE ===")

df_citibike_geo.groupBy(
    "start_geometry_type",
    "end_geometry_type"
).agg(
    count("*").alias("total")
).orderBy(
    "start_geometry_type",
    "end_geometry_type"
).show(
    truncate=False
)


print("=== SRID CITIBIKE ===")

df_citibike_geo.groupBy(
    "start_srid",
    "end_srid"
).agg(
    count("*").alias("total")
).orderBy(
    "start_srid",
    "end_srid"
).show()

+--------------+----------------+--------------+--------------------+------------------+--------------------+------------------+
|total_citibike|start_point_null|end_point_null|start_geometry_types|end_geometry_types|start_srid_distintos|end_srid_distintos|
+--------------+----------------+--------------+--------------------+------------------+--------------------+------------------+
|       5676020|               0|            74|                   1|                 1|                   1|                 1|
+--------------+----------------+--------------+--------------------+------------------+--------------------+------------------+

=== TIPOS DE GEOMETRÍA CITIBIKE ===
+-------------------+-----------------+-------+
|start_geometry_type|end_geometry_type|total  |
+-------------------+-----------------+-------+
|ST_Point           |NULL             |74     |
|ST_Point           |ST_Point         |5675946|
+-------------------+-----------------+-------+

=== SRID CITIBIKE ===
+------

In [0]:
df_zones_start = (
    df_taxi_zone_features_geo
    .select(
        col("objectid")
            .alias("start_zone_objectid"),

        col("location_id")
            .alias("start_location_id"),

        col("zone")
            .alias("start_zone"),

        col("borough")
            .alias("start_borough"),

        col("borough_category")
            .alias("start_borough_category"),

        col("zone_geometry")
    )
)


df_citibike_start_zone = (
    df_citibike_geo.alias("b")

    .join(
        F.broadcast(
            df_zones_start.alias("z")
        ),

        dbf.st_covers(
            col("z.zone_geometry"),
            col("b.start_point")
        ),

        "left"
    )

    .select(
        col("b.*"),

        col("z.start_zone_objectid"),
        col("z.start_location_id"),
        col("z.start_zone"),
        col("z.start_borough"),
        col("z.start_borough_category")
    )
)

In [0]:
# Persistir para evitar recalcular el Spatial Join
df_citibike_start_zone.persist()


# ============================================================
# MATERIALIZAR CACHE
# ============================================================

total_filas_join = (
    df_citibike_start_zone
    .count()
)


# ============================================================
# AGRUPAR UNA SOLA VEZ POR VIAJE
# ============================================================

df_start_matches = (
    df_citibike_start_zone

    .groupBy("trip_id")

    .agg(
        count(
            "start_zone_objectid"
        ).alias("zone_matches")
    )
)


# ============================================================
# CALCULAR TODAS LAS MÉTRICAS EN UNA SOLA AGREGACIÓN
# ============================================================

start_stats = (
    df_start_matches

    .agg(
        count("*")
            .alias("trip_ids_distintos"),

        sum(
            when(
                col("zone_matches") == 0,
                lit(1)
            ).otherwise(lit(0))
        ).alias("sin_match"),

        sum(
            when(
                col("zone_matches") == 1,
                lit(1)
            ).otherwise(lit(0))
        ).alias("un_match"),

        sum(
            when(
                col("zone_matches") > 1,
                lit(1)
            ).otherwise(lit(0))
        ).alias("multiples_matches")
    )

    .first()
)


total_trip_ids_join = (
    start_stats["trip_ids_distintos"]
)

sin_match = (
    start_stats["sin_match"]
)

un_match = (
    start_stats["un_match"]
)

multiples_matches = (
    start_stats["multiples_matches"]
)


# En un LEFT JOIN:
# zone_matches = 0 equivale a no encontrar Start Zone
sin_start_zone = sin_match


print(
    "=== VALIDACIÓN SPATIAL JOIN START CITIBIKE ==="
)

print(
    f"Citi Bike original       : "
    f"{total_citibike_original:,}"
)

print(
    f"Filas después join       : "
    f"{total_filas_join:,}"
)

print(
    f"Trip IDs distintos       : "
    f"{total_trip_ids_join:,}"
)

print(
    f"Sin Start Zone           : "
    f"{sin_start_zone:,}"
)

print()

print(
    "=== CANTIDAD DE MATCHES POR VIAJE ==="
)

print(
    f"Sin match                : "
    f"{sin_match:,}"
)

print(
    f"Con 1 match              : "
    f"{un_match:,}"
)

print(
    f"Con >1 match             : "
    f"{multiples_matches:,}"
)


# ============================================================
# LIBERAR CACHE
# ============================================================

df_citibike_start_zone.unpersist()

=== VALIDACIÓN SPATIAL JOIN START CITIBIKE ===
Citi Bike original       : 5,676,020
Filas después join       : 5,676,020
Trip IDs distintos       : 5,676,020
Sin Start Zone           : 0

=== CANTIDAD DE MATCHES POR VIAJE ===
Sin match                : 0
Con 1 match              : 5,676,020
Con >1 match             : 0


In [0]:
df_zones_end = (
    df_taxi_zone_features_geo

    .select(
        col("objectid")
            .alias("end_zone_objectid"),

        col("location_id")
            .alias("end_location_id"),

        col("zone")
            .alias("end_zone"),

        col("borough")
            .alias("end_borough"),

        col("borough_category")
            .alias("end_borough_category"),

        col("zone_geometry")
    )
)


# ============================================================
# DATAFRAME MÍNIMO PARA EL SPATIAL JOIN
# ============================================================

df_citibike_end_points = (
    df_citibike_geo

    .select(
        col("trip_id"),
        col("end_point")
    )
)


# ============================================================
# SPATIAL JOIN
# ============================================================

df_citibike_end_zone = (
    df_citibike_end_points.alias("b")

    .join(
        F.broadcast(
            df_zones_end.alias("z")
        ),

        dbf.st_covers(
            col("z.zone_geometry"),
            col("b.end_point")
        ),

        "left"
    )

    .select(
        col("b.trip_id"),

        col("z.end_zone_objectid"),
        col("z.end_location_id"),
        col("z.end_zone"),
        col("z.end_borough"),
        col("z.end_borough_category")
    )

    # ========================================================
    # PERSISTIR PARA LAS VALIDACIONES SIGUIENTES
    # ========================================================

    .persist()
)

In [0]:
# ============================================================
# CELDA 83 - VALIDAR SPATIAL JOIN END CITIBIKE
# OPTIMIZADO
# ============================================================

total_citibike_original = (
    df_citibike_geo
    .count()
)


# ============================================================
# ESTA ACCIÓN MATERIALIZA EL DATAFRAME PERSISTIDO
# ============================================================

total_filas_join = (
    df_citibike_end_zone
    .count()
)


# ============================================================
# MATCHES POR VIAJE
# ============================================================

df_end_matches = (
    df_citibike_end_zone

    .groupBy("trip_id")

    .agg(
        count(
            "end_zone_objectid"
        ).alias("zone_matches")
    )
)


# ============================================================
# TODAS LAS MÉTRICAS EN UNA SOLA AGREGACIÓN
# ============================================================

end_stats = (
    df_end_matches

    .agg(
        count("*")
            .alias("trip_ids_distintos"),

        sum(
            when(
                col("zone_matches") == 0,
                lit(1)
            ).otherwise(lit(0))
        ).alias("sin_match"),

        sum(
            when(
                col("zone_matches") == 1,
                lit(1)
            ).otherwise(lit(0))
        ).alias("un_match"),

        sum(
            when(
                col("zone_matches") > 1,
                lit(1)
            ).otherwise(lit(0))
        ).alias("multiples_matches")
    )

    .first()
)


total_trip_ids_join = (
    end_stats["trip_ids_distintos"]
)

sin_match = (
    end_stats["sin_match"]
)

un_match = (
    end_stats["un_match"]
)

multiples_matches = (
    end_stats["multiples_matches"]
)


sin_end_zone = sin_match


print(
    "=== VALIDACIÓN SPATIAL JOIN END CITIBIKE ==="
)

print(
    f"Citi Bike original       : "
    f"{total_citibike_original:,}"
)

print(
    f"Filas después join       : "
    f"{total_filas_join:,}"
)

print(
    f"Trip IDs distintos       : "
    f"{total_trip_ids_join:,}"
)

print(
    f"Sin End Zone             : "
    f"{sin_end_zone:,}"
)

print()

print(
    "=== CANTIDAD DE MATCHES POR VIAJE ==="
)

print(
    f"Sin match                : "
    f"{sin_match:,}"
)

print(
    f"Con 1 match              : "
    f"{un_match:,}"
)

print(
    f"Con >1 match             : "
    f"{multiples_matches:,}"
)

=== VALIDACIÓN SPATIAL JOIN END CITIBIKE ===
Citi Bike original       : 5,676,020
Filas después join       : 5,676,020
Trip IDs distintos       : 5,676,020
Sin End Zone             : 106

=== CANTIDAD DE MATCHES POR VIAJE ===
Sin match                : 106
Con 1 match              : 5,675,914
Con >1 match             : 0


In [0]:
# CONSTRUIR CITIBIKE ENRIQUECIDO

df_citibike_geo_transform = (
    df_citibike_start_zone.alias("b")

    .join(
        df_citibike_end_zone.alias("e"),

        col("b.trip_id") == col("e.trip_id"),

        "left"
    )

    .select(
        col("b.*"),

        col("e.end_zone_objectid"),
        col("e.end_location_id"),
        col("e.end_zone"),
        col("e.end_borough"),
        col("e.end_borough_category")
    )

    # ============================================================
    # DQ START ZONE
    # ============================================================

    .withColumn(
        "dq_missing_start_zone",
        when(
            col("start_location_id").isNull(),
            lit(1)
        ).otherwise(lit(0))
    )

    # ============================================================
    # DQ END ZONE
    # ============================================================

    .withColumn(
        "dq_missing_end_zone",
        when(
            col("end_location_id").isNull(),
            lit(1)
        ).otherwise(lit(0))
    )

    # ============================================================
    # VALIDEZ GEOESPACIAL
    # ============================================================

    .withColumn(
        "dq_valid_spatial",
        when(
            col("start_location_id").isNotNull()
            & col("end_location_id").isNotNull(),
            lit(1)
        ).otherwise(lit(0))
    )

    # ============================================================
    # PERSISTIR RESULTADO INTEGRADO
    # ============================================================

    .persist()
)

In [0]:
# VALIDAR DQ GEOESPACIAL CITIBIKE
# ============================================================

total_citibike_geo = (
    df_citibike_geo_transform
    .count()
)


citibike_geo_stats = (
    df_citibike_geo_transform

    .agg(
        count("*")
            .alias("total_registros"),

        countDistinct("trip_id")
            .alias("trip_ids_unicos"),

        sum("dq_missing_start_zone")
            .alias("missing_start_zone"),

        sum("dq_missing_end_zone")
            .alias("missing_end_zone"),

        sum(
            when(
                col("dq_valid_spatial") == 1,
                lit(1)
            ).otherwise(lit(0))
        ).alias("valid_spatial"),

        sum(
            when(
                (col("dq_missing_start_zone") == 1)
                & (col("dq_missing_end_zone") == 0),
                lit(1)
            ).otherwise(lit(0))
        ).alias("start_missing_only"),

        sum(
            when(
                (col("dq_missing_start_zone") == 0)
                & (col("dq_missing_end_zone") == 1),
                lit(1)
            ).otherwise(lit(0))
        ).alias("end_missing_only"),

        sum(
            when(
                (col("dq_missing_start_zone") == 1)
                & (col("dq_missing_end_zone") == 1),
                lit(1)
            ).otherwise(lit(0))
        ).alias("both_missing")
    )
)


citibike_geo_stats.show(
    truncate=False
)

+---------------+---------------+------------------+----------------+-------------+------------------+----------------+------------+
|total_registros|trip_ids_unicos|missing_start_zone|missing_end_zone|valid_spatial|start_missing_only|end_missing_only|both_missing|
+---------------+---------------+------------------+----------------+-------------+------------------+----------------+------------+
|5676020        |5676020        |0                 |106             |5675914      |0                 |106             |0           |
+---------------+---------------+------------------+----------------+-------------+------------------+----------------+------------+



In [0]:
# ============================================================
# PREPARAR CITIBIKE ENRIQUECIDO PARA SILVER
# ============================================================

df_citibike_enriched_silver = (
    df_citibike_geo_transform

    .select(
        # =====================================================
        # DATOS CITIBIKE
        # =====================================================

        col("trip_id"),
        col("tripduration"),
        col("starttime"),
        col("stoptime"),

        col("start_station_id"),
        col("start_station_name"),
        col("start_station_latitude"),
        col("start_station_longitude"),

        col("end_station_id"),
        col("end_station_name"),
        col("end_station_latitude"),
        col("end_station_longitude"),

        col("bikeid"),
        col("usertype"),
        col("birth_year"),
        col("gender"),

        # =====================================================
        # METADATA
        # =====================================================

        col("_ingestion_timestamp"),
        col("_source_file"),
        col("_source_system"),
        col("_batch_id"),

        # =====================================================
        # DQ ORIGINAL
        # =====================================================

        col("dq_invalid_duration"),
        col("dq_duration_mismatch"),
        col("dq_invalid_start_coordinates"),
        col("dq_invalid_end_coordinates"),
        col("dq_missing_birth_year"),
        col("dq_age_outlier"),
        col("dq_long_duration"),
        col("dq_valid_trip"),

        # =====================================================
        # DERIVADAS
        # =====================================================

        col("duration_minutes"),
        col("duration_hours"),

        col("start_date"),
        col("start_hour"),
        col("start_day_of_week"),
        col("start_month"),
        col("start_year"),
        col("is_weekend"),

        col("rider_age"),

        # =====================================================
        # START TAXI ZONE
        # =====================================================

        col("start_zone_objectid"),
        col("start_location_id"),
        col("start_zone"),
        col("start_borough"),
        col("start_borough_category"),

        # =====================================================
        # END TAXI ZONE
        # =====================================================

        col("end_zone_objectid"),
        col("end_location_id"),
        col("end_zone"),
        col("end_borough"),
        col("end_borough_category"),

        # =====================================================
        # DQ GEOESPACIAL
        # =====================================================

        col("dq_missing_start_zone"),
        col("dq_missing_end_zone"),
        col("dq_valid_spatial")
    )
)


print(
    f"Columnas Citi Bike enriquecido: "
    f"{len(df_citibike_enriched_silver.columns)}"
)

print(
    f"Registros Citi Bike enriquecido: "
    f"{df_citibike_enriched_silver.count():,}"
)

Columnas Citi Bike enriquecido: 50
Registros Citi Bike enriquecido: 5,676,020


In [0]:
# ============================================================
# VALIDAR CONTRATO CITIBIKE TRIPS ENRICHED
# ============================================================

tabla_citibike_enriched_silver = (
    f"{catalogo}.{esquema_sink}.citibike_trips_enriched"
)


columnas_df = (
    df_citibike_enriched_silver
    .columns
)

columnas_tabla = (
    spark.table(
        tabla_citibike_enriched_silver
    )
    .columns
)


print(
    f"Columnas DataFrame : {len(columnas_df)}"
)

print(
    f"Columnas tabla     : {len(columnas_tabla)}"
)


if columnas_df != columnas_tabla:
    raise Exception(
        "El orden o nombre de las columnas de "
        "Citi Bike Trips Enriched no coincide."
    )


schema_df = [
    (
        field.name,
        field.dataType.simpleString()
    )
    for field
    in df_citibike_enriched_silver.schema.fields
]


schema_tabla = [
    (
        field.name,
        field.dataType.simpleString()
    )
    for field
    in spark.table(
        tabla_citibike_enriched_silver
    ).schema.fields
]


if schema_df != schema_tabla:
    raise Exception(
        "Los tipos de datos de Citi Bike Trips Enriched "
        "no coinciden con la tabla Silver."
    )


print(
    "Contrato Citi Bike Trips Enriched correcto."
)

Columnas DataFrame : 50
Columnas tabla     : 50
Contrato Citi Bike Trips Enriched correcto.


In [0]:
# ============================================================
#  ESCRIBIR CITIBIKE TRIPS ENRICHED
# ============================================================

df_citibike_enriched_silver.write\
    .mode("overwrite")\
    .insertInto(
        tabla_citibike_enriched_silver
    )

In [0]:
# ============================================================
# CELDA 89 - RECONCILIACIÓN Y DQ CITIBIKE ENRICHED
# ============================================================

df_citibike_enriched_check = (
    spark.table(
        tabla_citibike_enriched_silver
    )
)


total_citibike_silver = (
    spark.table(
        tabla_citibike_silver
    )
    .count()
)


total_citibike_enriched = (
    df_citibike_enriched_check
    .count()
)


ids_citibike_enriched = (
    df_citibike_enriched_check
    .select("trip_id")
    .distinct()
    .count()
)


print("=== RECONCILIACIÓN CITIBIKE ENRICHED ===")

print(
    f"Citi Bike Silver original : "
    f"{total_citibike_silver:,}"
)

print(
    f"Citi Bike Enriched Silver : "
    f"{total_citibike_enriched:,}"
)

print(
    f"Trip IDs únicos           : "
    f"{ids_citibike_enriched:,}"
)


if total_citibike_silver != total_citibike_enriched:
    raise Exception(
        "La cantidad de registros Citi Bike "
        "Silver y Enriched no coincide."
    )


if total_citibike_enriched != ids_citibike_enriched:
    raise Exception(
        "Citi Bike Enriched contiene Trip IDs duplicados."
    )


print()
print("=== DQ CITIBIKE ENRICHED ===")


df_citibike_enriched_check.agg(

    sum("dq_invalid_duration")
        .alias("invalid_duration"),

    sum("dq_duration_mismatch")
        .alias("duration_mismatch"),

    sum("dq_invalid_start_coordinates")
        .alias("invalid_start_coordinates"),

    sum("dq_invalid_end_coordinates")
        .alias("invalid_end_coordinates"),

    sum("dq_missing_birth_year")
        .alias("missing_birth_year"),

    sum("dq_age_outlier")
        .alias("age_outlier"),

    sum("dq_long_duration")
        .alias("long_duration"),

    sum(
        when(
            col("dq_valid_trip") == 1,
            lit(1)
        ).otherwise(lit(0))
    ).alias("valid_trip"),

    sum("dq_missing_start_zone")
        .alias("missing_start_zone"),

    sum("dq_missing_end_zone")
        .alias("missing_end_zone"),

    sum(
        when(
            col("dq_valid_spatial") == 1,
            lit(1)
        ).otherwise(lit(0))
    ).alias("valid_spatial")

).show(
    truncate=False
)

=== RECONCILIACIÓN CITIBIKE ENRICHED ===
Citi Bike Silver original : 5,676,020
Citi Bike Enriched Silver : 5,676,020
Trip IDs únicos           : 5,676,020

=== DQ CITIBIKE ENRICHED ===
+----------------+-----------------+-------------------------+-----------------------+------------------+-----------+-------------+----------+------------------+----------------+-------------+
|invalid_duration|duration_mismatch|invalid_start_coordinates|invalid_end_coordinates|missing_birth_year|age_outlier|long_duration|valid_trip|missing_start_zone|missing_end_zone|valid_spatial|
+----------------+-----------------+-------------------------+-----------------------+------------------+-----------+-------------+----------+------------------+----------------+-------------+
|0               |127              |0                        |74                     |649611            |2075       |13466        |5675946   |0                 |106             |5675914      |
+----------------+-----------------+-------

In [0]:
# ============================================================
# LIBERAR CACHE CITIBIKE
# ============================================================

df_citibike_end_zone.unpersist()

df_citibike_geo_transform.unpersist()


print(
    "Cache Citi Bike liberado correctamente."
)

Cache Citi Bike liberado correctamente.


In [0]:
# ============================================================
# VALIDAR TIMEZONE ANTES DE INTEGRAR WEATHER
# ============================================================

tabla_weather_silver = (
    f"{catalogo}.{esquema_sink}.weather_hourly"
)

tabla_taxi_enriched_silver = (
    f"{catalogo}.{esquema_sink}.taxi_trips_enriched"
)

tabla_citibike_enriched_silver = (
    f"{catalogo}.{esquema_sink}.citibike_trips_enriched"
)


# ============================================================
# TIMEZONE DE LA SESIÓN SPARK
# ============================================================

session_timezone = (
    spark.conf.get(
        "spark.sql.session.timeZone"
    )
)


print(
    "=== TIMEZONE SPARK ==="
)

print(
    f"spark.sql.session.timeZone: "
    f"{session_timezone}"
)


# ============================================================
# RANGOS TEMPORALES
# ============================================================

print()
print(
    "=== RANGO WEATHER ==="
)

spark.table(
    tabla_weather_silver
).agg(
    min("time").alias("min_time"),
    max("time").alias("max_time"),
    count("*").alias("registros")
).show(
    truncate=False
)


print(
    "=== RANGO TAXI ==="
)

spark.table(
    tabla_taxi_enriched_silver
).agg(
    min("pickup_datetime")
        .alias("min_pickup"),

    max("pickup_datetime")
        .alias("max_pickup"),

    count("*")
        .alias("registros")
).show(
    truncate=False
)


print(
    "=== RANGO CITIBIKE ==="
)

spark.table(
    tabla_citibike_enriched_silver
).agg(
    min("starttime")
        .alias("min_start"),

    max("starttime")
        .alias("max_start"),

    count("*")
        .alias("registros")
).show(
    truncate=False
)


# ============================================================
# METADATA WEATHER
# ============================================================

print(
    "=== SOURCE WEATHER ==="
)

spark.table(
    tabla_weather_silver
).select(
    "_source_file",
    "_source_system"
).distinct().show(
    truncate=False
)


# ============================================================
# PRIMERAS HORAS WEATHER
# ============================================================

print(
    "=== PRIMERAS 12 HORAS WEATHER ==="
)

spark.table(
    tabla_weather_silver
).select(
    "time",
    "temperature_2m_c",
    "precipitation_mm",
    "rain_mm",
    "weather_condition"
).orderBy(
    "time"
).show(
    12,
    truncate=False
)

=== TIMEZONE SPARK ===
spark.sql.session.timeZone: Etc/UTC

=== RANGO WEATHER ===
+-------------------+-------------------+---------+
|min_time           |max_time           |registros|
+-------------------+-------------------+---------+
|2016-01-01 00:00:00|2016-06-30 23:00:00|4368     |
+-------------------+-------------------+---------+

=== RANGO TAXI ===
+-------------------+-------------------+---------+
|min_pickup         |max_pickup         |registros|
+-------------------+-------------------+---------+
|2016-01-01 00:00:17|2016-06-30 23:59:39|1458644  |
+-------------------+-------------------+---------+

=== RANGO CITIBIKE ===
+-------------------+-------------------+---------+
|min_start          |max_start          |registros|
+-------------------+-------------------+---------+
|2016-01-01 00:00:41|2016-06-30 23:59:58|5676020  |
+-------------------+-------------------+---------+

=== SOURCE WEATHER ===
+---------------------------------------------------------------------

In [0]:
# ============================================================
# PREPARAR WEATHER PARA INTEGRACIÓN
# ============================================================

df_weather_integration = (
    spark.table(
        tabla_weather_silver
    )

    .filter(
        col("dq_valid_weather") == 1
    )

    .select(
        col("time")
            .alias("weather_hour"),

        col("temperature_2m_c"),
        col("precipitation_mm"),
        col("rain_mm"),
        col("cloudcover_pct"),
        col("windspeed_10m_kmh"),
        col("winddirection_10m_deg"),
        col("weather_condition"),
        col("has_precipitation"),
        col("has_rain"),
        col("dq_valid_weather")
    )
)


print("=== WEATHER PARA INTEGRACIÓN ===")

df_weather_integration.agg(
    count("*")
        .alias("registros"),

    countDistinct("weather_hour")
        .alias("horas_unicas"),

    min("weather_hour")
        .alias("min_weather_hour"),

    max("weather_hour")
        .alias("max_weather_hour")
).show(
    truncate=False
)

=== WEATHER PARA INTEGRACIÓN ===
+---------+------------+-------------------+-------------------+
|registros|horas_unicas|min_weather_hour   |max_weather_hour   |
+---------+------------+-------------------+-------------------+
|4368     |4368        |2016-01-01 00:00:00|2016-06-30 23:00:00|
+---------+------------+-------------------+-------------------+



In [0]:
# ============================================================
# INTEGRAR TAXI CON WEATHER
# ============================================================

df_taxi_weather = (
    spark.table(
        tabla_taxi_enriched_silver
    )

    .withColumn(
        "event_hour",
        date_trunc(
            "hour",
            col("pickup_datetime")
        )
    )

    .alias("t")

    .join(
        F.broadcast(
            df_weather_integration.alias("w")
        ),

        col("t.event_hour")
        == col("w.weather_hour"),

        "left"
    )

    .select(
        col("t.*"),

        col("w.temperature_2m_c"),
        col("w.precipitation_mm"),
        col("w.rain_mm"),
        col("w.cloudcover_pct"),
        col("w.windspeed_10m_kmh"),
        col("w.winddirection_10m_deg"),
        col("w.weather_condition"),
        col("w.has_precipitation"),
        col("w.has_rain"),

        when(
            col("w.weather_hour").isNull(),
            lit(1)
        ).otherwise(
            lit(0)
        ).alias("dq_missing_weather")
    )
)

In [0]:
# ============================================================
# INTEGRAR CITIBIKE CON WEATHER
# ============================================================

df_citibike_weather = (
    spark.table(
        tabla_citibike_enriched_silver
    )

    .withColumn(
        "event_hour",
        date_trunc(
            "hour",
            col("starttime")
        )
    )

    .alias("b")

    .join(
        F.broadcast(
            df_weather_integration.alias("w")
        ),

        col("b.event_hour")
        == col("w.weather_hour"),

        "left"
    )

    .select(
        col("b.*"),

        col("w.temperature_2m_c"),
        col("w.precipitation_mm"),
        col("w.rain_mm"),
        col("w.cloudcover_pct"),
        col("w.windspeed_10m_kmh"),
        col("w.winddirection_10m_deg"),
        col("w.weather_condition"),
        col("w.has_precipitation"),
        col("w.has_rain"),

        when(
            col("w.weather_hour").isNull(),
            lit(1)
        ).otherwise(
            lit(0)
        ).alias("dq_missing_weather")
    )
)


print(
    f"Columnas Taxi + Weather     : "
    f"{len(df_taxi_weather.columns)}"
)

print(
    f"Columnas Citi Bike + Weather: "
    f"{len(df_citibike_weather.columns)}"
)

Columnas Taxi + Weather     : 53
Columnas Citi Bike + Weather: 61


In [0]:
# ============================================================
# VALIDAR INTEGRACIÓN WEATHER
# ============================================================

taxi_weather_stats = (
    df_taxi_weather

    .agg(
        count("*")
            .alias("total_registros"),

        countDistinct("id")
            .alias("ids_unicos"),

        sum("dq_missing_weather")
            .alias("missing_weather"),

        min("event_hour")
            .alias("min_event_hour"),

        max("event_hour")
            .alias("max_event_hour")
    )

    .first()
)


citibike_weather_stats = (
    df_citibike_weather

    .agg(
        count("*")
            .alias("total_registros"),

        countDistinct("trip_id")
            .alias("trip_ids_unicos"),

        sum("dq_missing_weather")
            .alias("missing_weather"),

        min("event_hour")
            .alias("min_event_hour"),

        max("event_hour")
            .alias("max_event_hour")
    )

    .first()
)


print("=== TAXI + WEATHER ===")

print(
    f"Total registros : "
    f"{taxi_weather_stats['total_registros']:,}"
)

print(
    f"IDs únicos      : "
    f"{taxi_weather_stats['ids_unicos']:,}"
)

print(
    f"Missing Weather : "
    f"{taxi_weather_stats['missing_weather']:,}"
)

print(
    f"Hora mínima     : "
    f"{taxi_weather_stats['min_event_hour']}"
)

print(
    f"Hora máxima     : "
    f"{taxi_weather_stats['max_event_hour']}"
)


print()
print("=== CITIBIKE + WEATHER ===")

print(
    f"Total registros : "
    f"{citibike_weather_stats['total_registros']:,}"
)

print(
    f"Trip IDs únicos : "
    f"{citibike_weather_stats['trip_ids_unicos']:,}"
)

print(
    f"Missing Weather : "
    f"{citibike_weather_stats['missing_weather']:,}"
)

print(
    f"Hora mínima     : "
    f"{citibike_weather_stats['min_event_hour']}"
)

print(
    f"Hora máxima     : "
    f"{citibike_weather_stats['max_event_hour']}"
)

=== TAXI + WEATHER ===
Total registros : 1,458,644
IDs únicos      : 1,458,644
Missing Weather : 0
Hora mínima     : 2016-01-01 00:00:00
Hora máxima     : 2016-06-30 23:00:00

=== CITIBIKE + WEATHER ===
Total registros : 5,676,020
Trip IDs únicos : 5,676,020
Missing Weather : 0
Hora mínima     : 2016-01-01 00:00:00
Hora máxima     : 2016-06-30 23:00:00


In [0]:
# ============================================================
# NORMALIZAR TAXI A MOBILITY EVENTS
# ============================================================

df_taxi_mobility = (
    df_taxi_weather

    .select(
        # =====================================================
        # IDENTIFICACIÓN
        # =====================================================

        concat(
            lit("TAXI_"),
            col("id")
        ).alias("event_id"),

        col("id")
            .alias("source_event_id"),

        lit("TAXI")
            .alias("transport_type"),

        # =====================================================
        # TIEMPO
        # =====================================================

        col("pickup_datetime")
            .alias("start_datetime"),

        col("dropoff_datetime")
            .alias("end_datetime"),

        col("trip_duration")
            .cast("long")
            .alias("duration_seconds"),

        col("duration_minutes"),
        col("duration_hours"),

        # =====================================================
        # COORDENADAS
        # =====================================================

        col("pickup_latitude")
            .alias("start_latitude"),

        col("pickup_longitude")
            .alias("start_longitude"),

        col("dropoff_latitude")
            .alias("end_latitude"),

        col("dropoff_longitude")
            .alias("end_longitude"),

        # =====================================================
        # START ZONE
        # =====================================================

        col("pickup_location_id")
            .alias("start_location_id"),

        col("pickup_zone")
            .alias("start_zone"),

        col("pickup_borough")
            .alias("start_borough"),

        col("pickup_borough_category")
            .alias("start_borough_category"),

        # =====================================================
        # END ZONE
        # =====================================================

        col("dropoff_location_id")
            .alias("end_location_id"),

        col("dropoff_zone")
            .alias("end_zone"),

        col("dropoff_borough")
            .alias("end_borough"),

        col("dropoff_borough_category")
            .alias("end_borough_category"),

        # =====================================================
        # VARIABLES TEMPORALES
        # =====================================================

        col("pickup_date")
            .alias("event_date"),

        col("event_hour")
            .alias("event_hour_ts"),

        col("pickup_hour")
            .alias("event_hour"),

        col("pickup_day_of_week")
            .alias("event_day_of_week"),

        col("pickup_month")
            .alias("event_month"),

        col("pickup_year")
            .alias("event_year"),

        col("is_weekend"),

        # =====================================================
        # WEATHER
        # =====================================================

        col("temperature_2m_c"),
        col("precipitation_mm"),
        col("rain_mm"),
        col("cloudcover_pct"),
        col("windspeed_10m_kmh"),
        col("winddirection_10m_deg"),
        col("weather_condition"),
        col("has_precipitation"),
        col("has_rain"),

        # =====================================================
        # DQ COMÚN
        # =====================================================

        col("dq_invalid_duration"),

        col("dq_invalid_pickup_coordinates")
            .alias("dq_invalid_start_coordinates"),

        col("dq_invalid_dropoff_coordinates")
            .alias("dq_invalid_end_coordinates"),

        col("dq_valid_spatial"),
        col("dq_missing_weather"),

        when(
            (col("dq_invalid_duration") == 0)
            & (col("dq_invalid_pickup_coordinates") == 0)
            & (col("dq_invalid_dropoff_coordinates") == 0)
            & (col("dq_valid_spatial") == 1)
            & (col("dq_missing_weather") == 0),
            lit(1)
        ).otherwise(
            lit(0)
        ).alias("dq_valid_event"),

        # =====================================================
        # ATRIBUTOS ESPECÍFICOS
        # =====================================================

        col("passenger_count"),

        lit(None)
            .cast("string")
            .alias("usertype"),

        lit(None)
            .cast("int")
            .alias("rider_age"),

        # =====================================================
        # METADATA
        # =====================================================

        col("_ingestion_timestamp"),
        col("_source_file"),
        col("_source_system"),
        col("_batch_id")
    )
)

In [0]:
# ============================================================
# NORMALIZAR CITIBIKE A MOBILITY EVENTS
# ============================================================

df_citibike_mobility = (
    df_citibike_weather

    .select(
        # =====================================================
        # IDENTIFICACIÓN
        # =====================================================

        concat(
            lit("BIKE_"),
            col("trip_id")
        ).alias("event_id"),

        col("trip_id")
            .alias("source_event_id"),

        lit("BIKE")
            .alias("transport_type"),

        # =====================================================
        # TIEMPO
        # =====================================================

        col("starttime")
            .alias("start_datetime"),

        col("stoptime")
            .alias("end_datetime"),

        col("tripduration")
            .cast("long")
            .alias("duration_seconds"),

        col("duration_minutes"),
        col("duration_hours"),

        # =====================================================
        # COORDENADAS
        # =====================================================

        col("start_station_latitude")
            .alias("start_latitude"),

        col("start_station_longitude")
            .alias("start_longitude"),

        col("end_station_latitude")
            .alias("end_latitude"),

        col("end_station_longitude")
            .alias("end_longitude"),

        # =====================================================
        # START ZONE
        # =====================================================

        col("start_location_id"),
        col("start_zone"),
        col("start_borough"),
        col("start_borough_category"),

        # =====================================================
        # END ZONE
        # =====================================================

        col("end_location_id"),
        col("end_zone"),
        col("end_borough"),
        col("end_borough_category"),

        # =====================================================
        # VARIABLES TEMPORALES
        # =====================================================

        col("start_date")
            .alias("event_date"),

        col("event_hour")
            .alias("event_hour_ts"),

        col("start_hour")
            .alias("event_hour"),

        col("start_day_of_week")
            .alias("event_day_of_week"),

        col("start_month")
            .alias("event_month"),

        col("start_year")
            .alias("event_year"),

        col("is_weekend"),

        # =====================================================
        # WEATHER
        # =====================================================

        col("temperature_2m_c"),
        col("precipitation_mm"),
        col("rain_mm"),
        col("cloudcover_pct"),
        col("windspeed_10m_kmh"),
        col("winddirection_10m_deg"),
        col("weather_condition"),
        col("has_precipitation"),
        col("has_rain"),

        # =====================================================
        # DQ COMÚN
        # =====================================================

        col("dq_invalid_duration"),
        col("dq_invalid_start_coordinates"),
        col("dq_invalid_end_coordinates"),
        col("dq_valid_spatial"),
        col("dq_missing_weather"),

        when(
            (col("dq_invalid_duration") == 0)
            & (col("dq_invalid_start_coordinates") == 0)
            & (col("dq_invalid_end_coordinates") == 0)
            & (col("dq_valid_spatial") == 1)
            & (col("dq_missing_weather") == 0),
            lit(1)
        ).otherwise(
            lit(0)
        ).alias("dq_valid_event"),

        # =====================================================
        # ATRIBUTOS ESPECÍFICOS
        # =====================================================

        lit(None)
            .cast("int")
            .alias("passenger_count"),

        col("usertype"),

        col("rider_age"),

        # =====================================================
        # METADATA
        # =====================================================

        col("_ingestion_timestamp"),
        col("_source_file"),
        col("_source_system"),
        col("_batch_id")
    )
)

In [0]:
# ============================================================
# CONSTRUIR MOBILITY EVENTS
# ============================================================

df_mobility_events = (
    df_taxi_mobility

    .unionByName(
        df_citibike_mobility
    )

    .persist()
)


total_mobility_events = (
    df_mobility_events
    .count()
)


print(
    f"Mobility Events materializados: "
    f"{total_mobility_events:,}"
)

print(
    f"Columnas Mobility Events      : "
    f"{len(df_mobility_events.columns)}"
)

Mobility Events materializados: 7,134,664
Columnas Mobility Events      : 49


In [0]:
# ============================================================
# VALIDAR MOBILITY EVENTS
# ============================================================

df_mobility_events.groupBy(
    "transport_type"
).agg(
    count("*")
        .alias("total_events"),

    countDistinct("event_id")
        .alias("event_ids_unicos"),

    sum(
        when(
            col("dq_valid_event") == 1,
            lit(1)
        ).otherwise(lit(0))
    ).alias("valid_events"),

    sum(
        when(
            col("dq_valid_event") == 0,
            lit(1)
        ).otherwise(lit(0))
    ).alias("invalid_events"),

    sum("dq_missing_weather")
        .alias("missing_weather")
).orderBy(
    "transport_type"
).show(
    truncate=False
)


print("=== VALIDACIÓN GLOBAL ===")

df_mobility_events.agg(
    count("*")
        .alias("total_events"),

    countDistinct("event_id")
        .alias("event_ids_unicos"),

    countDistinct("source_event_id", "transport_type")
        .alias("source_ids_unicos"),

    sum(
        when(
            col("dq_valid_event") == 1,
            lit(1)
        ).otherwise(lit(0))
    ).alias("valid_events"),

    sum(
        when(
            col("dq_valid_event") == 0,
            lit(1)
        ).otherwise(lit(0))
    ).alias("invalid_events"),

    min("start_datetime")
        .alias("min_start_datetime"),

    max("start_datetime")
        .alias("max_start_datetime")
).show(
    truncate=False
)

+--------------+------------+----------------+------------+--------------+---------------+
|transport_type|total_events|event_ids_unicos|valid_events|invalid_events|missing_weather|
+--------------+------------+----------------+------------+--------------+---------------+
|BIKE          |5676020     |5676020         |5675914     |106           |0              |
|TAXI          |1458644     |1458644         |1454641     |4003          |0              |
+--------------+------------+----------------+------------+--------------+---------------+

=== VALIDACIÓN GLOBAL ===
+------------+----------------+-----------------+------------+--------------+-------------------+-------------------+
|total_events|event_ids_unicos|source_ids_unicos|valid_events|invalid_events|min_start_datetime |max_start_datetime |
+------------+----------------+-----------------+------------+--------------+-------------------+-------------------+
|7134664     |7134664         |7134664          |7130555     |4109       

In [0]:
# ============================================================
# VALIDAR CONTRATO MOBILITY EVENTS
# ============================================================

tabla_mobility_events_silver = (
    f"{catalogo}.{esquema_sink}.mobility_events"
)


columnas_df = (
    df_mobility_events
    .columns
)

columnas_tabla = (
    spark.table(
        tabla_mobility_events_silver
    )
    .columns
)


print(
    f"Columnas DataFrame : {len(columnas_df)}"
)

print(
    f"Columnas tabla     : {len(columnas_tabla)}"
)


if columnas_df != columnas_tabla:
    raise Exception(
        "El orden o nombre de las columnas de "
        "Mobility Events no coincide con la tabla Silver."
    )


schema_df = [
    (
        field.name,
        field.dataType.simpleString()
    )
    for field
    in df_mobility_events.schema.fields
]


schema_tabla = [
    (
        field.name,
        field.dataType.simpleString()
    )
    for field
    in spark.table(
        tabla_mobility_events_silver
    ).schema.fields
]


if schema_df != schema_tabla:
    print(
        "=== SCHEMA DATAFRAME ==="
    )

    print(
        schema_df
    )

    print(
        "=== SCHEMA TABLA ==="
    )

    print(
        schema_tabla
    )

    raise Exception(
        "Los tipos de datos de Mobility Events "
        "no coinciden con la tabla Silver."
    )


print(
    "Contrato Mobility Events correcto."
)

Columnas DataFrame : 49
Columnas tabla     : 49
Contrato Mobility Events correcto.


In [0]:
# ============================================================
# ESCRIBIR MOBILITY EVENTS
# ============================================================

df_mobility_events.write\
    .mode("overwrite")\
    .insertInto(
        tabla_mobility_events_silver
    )

In [0]:
# ============================================================
# CELDA 102 - RECONCILIACIÓN FINAL MOBILITY EVENTS
# ============================================================

df_mobility_events_check = (
    spark.table(
        tabla_mobility_events_silver
    )
)


mobility_final_stats = (
    df_mobility_events_check

    .agg(
        count("*")
            .alias("total_events"),

        countDistinct("event_id")
            .alias("event_ids_unicos"),

        sum(
            when(
                col("transport_type") == "TAXI",
                lit(1)
            ).otherwise(lit(0))
        ).alias("taxi_events"),

        sum(
            when(
                col("transport_type") == "BIKE",
                lit(1)
            ).otherwise(lit(0))
        ).alias("bike_events"),

        sum(
            when(
                col("dq_valid_event") == 1,
                lit(1)
            ).otherwise(lit(0))
        ).alias("valid_events"),

        sum(
            when(
                col("dq_valid_event") == 0,
                lit(1)
            ).otherwise(lit(0))
        ).alias("invalid_events"),

        sum("dq_missing_weather")
            .alias("missing_weather")
    )
)


mobility_final_stats.show(
    truncate=False
)

+------------+----------------+-----------+-----------+------------+--------------+---------------+
|total_events|event_ids_unicos|taxi_events|bike_events|valid_events|invalid_events|missing_weather|
+------------+----------------+-----------+-----------+------------+--------------+---------------+
|7134664     |7134664         |1458644    |5676020    |7130555     |4109          |0              |
+------------+----------------+-----------+-----------+------------+--------------+---------------+



In [0]:
# ============================================================
# FINALIZAR TRANSFORM
# ============================================================

df_mobility_events.unpersist()


print(
    "Cache Mobility Events liberado correctamente."
)

print()
print(
    "============================================"
)

print(
    "TRANSFORMACIÓN SILVER FINALIZADA CORRECTAMENTE"
)

print(
    "============================================"
)

print()
print(
    "Tabla integrada:"
)

print(
    tabla_mobility_events_silver
)

Cache Mobility Events liberado correctamente.

TRANSFORMACIÓN SILVER FINALIZADA CORRECTAMENTE

Tabla integrada:
catalog_au.silver.mobility_events
